# 07 — Figures and tables

Two strictly separated sections.

**Section A — Primary (patient-level).** The patient is the biological replicate.
Everything in this section derives from `04_patient_level_inference.ipynb`:
rank-biserial correlations with 95 % bootstrap confidence intervals, Benjamini–Hochberg
adjusted q-values over the complete estimable endpoint family, and explicit accounting
of structural missingness. Findings are described as *associated with clinical-outcome
group*.

**Section B — Exploratory pooled-cell.** Descriptive maps and the pooled Good-vs-Poor
differential figures from `05_pooled_exploratory_analysis.ipynb`. Every figure in this
section carries the subtitle **"Exploratory pooled-cell analysis"**. Pooled p-values
count cell-type pairs, not patients, and are not patient-level evidence.

### Legacy code ported here

| source | change |
|---|---|
| `cpdb_plots_generation.py` (805 ln, canonical argparse version) | `--base-dir` replaced by the config cell; plot functions take an explicit context object so the script can be run three times (all-samples, pooled Good, pooled Poor). Label maps unchanged. |
| `plot_figures.py` | `INPUT_DIR = SCRIPT_DIR` replaced by explicit config paths; the 14 hardcoded `STALE` unlinks routed through `safe_unlink(path, allowed_root)`; the `grep` subprocess self-check replaced by an in-Python check. `ABBREV`, `C_GOOD` / `C_POOR` / `C_BOTH`, `STATUS_ORDER` kept. |
| `plot_consistency.py` | `BASE = os.path.dirname(os.path.abspath(__file__))` replaced by config paths. |

`cpdb_plots_generation_poor.py` is **not** ported — it is broken and superseded. Running
the canonical script for pooled Good as well as pooled Poor also removes the old
asymmetry where descriptive plots existed for Poor but never for Good.

The `cpdb_plots_generation.py` label maps were verified against `annot_6`:
`SHORT_NAMES` 19/19, `CELL_CATEGORIES` 19/19, `CELL_ORDER` 19/19 — no gaps, no extras.
No mapping rewrite is required, and an assertion below re-checks coverage against the
cell types actually present in each run.

### Deletion safety

This notebook can delete files only through `safe_unlink(path, allowed_root)`, and
`allowed_root` is always a directory inside `outputs/cellphoneDB/clinical_outcome_v2/`.
Nothing writes into `cpdb_results_*`, `t1/t2/t3_*`, or `cellphoneDB_copy_backup/`.

In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


In [ ]:
# =============================================================================
# CONFIGURATION — the only cell that defines paths or thresholds. Every
# analytical value is read from config.yaml, the single source of truth.
# =============================================================================
from pathlib import Path
import sys

NOTEBOOK = "07_figures_and_tables"

CODE_DIR    = Path(f"{ACC_DATA_ROOT}/scripts/named_scripts/"
                   "cellphoneDB_scripts/clinical_outcome_v2")
CONFIG_PATH = CODE_DIR / "config.yaml"

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
from utils import load_config                                        # noqa: E402

CFG = load_config(CONFIG_PATH)

PROJECT_ROOT = Path(CFG["project_root"])
OUTPUT_ROOT  = Path(CFG["output_root"])
assert Path(CFG["code_root"]) == CODE_DIR, \
    f"config code_root {CFG['code_root']} != {CODE_DIR}"

# --- Inputs (read-only) -----------------------------------------------------
CPDB_RUNS           = OUTPUT_ROOT / "cpdb_runs"
ALL_SAMPLES_DIR     = CPDB_RUNS / "all"
POOLED_GOOD_DIR     = CPDB_RUNS / "pooled" / "good"
POOLED_POOR_DIR     = CPDB_RUNS / "pooled" / "poor"
POOLED_SEEDS        = [int(s) for s in CFG["pooled_downsample_seeds"]]
POOLED_SEED_DIRS    = {s: CPDB_RUNS / f"pooled_seed{s}" for s in POOLED_SEEDS if s != 0}
PER_PATIENT_DIR     = CPDB_RUNS / "per_patient"

PRIMARY_DIR         = OUTPUT_ROOT / "primary"
PRIMARY_RESULTS_CSV = PRIMARY_DIR / "patient_level_primary_results.csv"
EXTRACT_DIR         = OUTPUT_ROOT / "inputs"          # 01_/03_ side tables
SENSITIVITY_DIR     = OUTPUT_ROOT / "sensitivity"     # 06_ outputs
EXPLORATORY_DIR     = OUTPUT_ROOT / "exploratory_pooled"
DE_RESULTS_DIR      = EXPLORATORY_DIR / "differential_enrichment_results"
CONSISTENCY_DIR     = EXPLORATORY_DIR / "consistency_results"

# --- Output figure/table tree (everything this notebook writes) -------------
FIG_ROOT            = OUTPUT_ROOT / "figures"
FIG_PRIMARY         = FIG_ROOT / "primary"
FIG_POOLED          = FIG_ROOT / "exploratory_pooled"
FIG_POOLED_ALL      = FIG_POOLED / "all_samples"
FIG_POOLED_GOOD     = FIG_POOLED / "pooled_good"
FIG_POOLED_POOR     = FIG_POOLED / "pooled_poor"
FIG_POOLED_DIFF     = FIG_POOLED / "differential"
FIG_POOLED_CONSIST  = FIG_POOLED / "consistency"
FIG_POOLED_SEEDS    = FIG_POOLED / "seed_stability"
TABLES_DIR          = OUTPUT_ROOT / "tables"

LOG_DIR             = OUTPUT_ROOT / "logs"
RUN_LOG             = LOG_DIR / f"{NOTEBOOK}_run_log.jsonl"
SUCCESS_JSON        = FIG_ROOT / "07_SUCCESS.json"
MANIFEST_PATH       = FIG_ROOT / "manifest.json"   # final MANIFEST.sha256 is written by 08_

# safe_unlink is never allowed outside this root.
DELETION_ALLOWED_ROOT = OUTPUT_ROOT

# --- Figure style -----------------------------------------------------------
DPI              = 600
FONT_SIZE        = 14
FIG_FORMATS      = ("png", "svg")
POOLED_SUBTITLE  = "Exploratory pooled-cell analysis"

# --- Thresholds — read from config.yaml, never re-declared ------------------
FDR_ALPHA        = float(CFG["fdr_alpha"])
MIN_CELLS        = int(CFG["patient_celltype_min_cells"])
SAMPLE_COL       = CFG["sample_column"]
CELL_TYPE_COL    = CFG["cell_type_column"]
OUTCOME_COL      = CFG["outcome_column"]
OUTCOME_LABELS   = list(CFG["outcomes"])
assert OUTCOME_COL == "clinical_outcome"
assert sorted(OUTCOME_LABELS) == ["Good", "Poor"]

# --- Presentation choices (not analytical; no config counterpart) -----------
N_FOREST           = 20   # top-N by q when there are no FDR hits
N_JITTER_ENDPOINTS = 6    # endpoints shown as patient-level jitter/box panels

# --- Expected cohort structure ----------------------------------------------
# Dataset facts established and asserted in 00_audit_inputs_and_sources, not
# configuration keys: config.yaml deliberately holds no cohort counts. They are
# re-asserted below against the derived patient mapping and the count matrix.
EXPECTED_N_GOOD      = 13
EXPECTED_N_POOR      = 11
EXPECTED_N_PATIENTS  = 24
EXPECTED_N_CELLTYPES = 19
assert EXPECTED_N_GOOD + EXPECTED_N_POOR == EXPECTED_N_PATIENTS

# --- Excel workbook ---------------------------------------------------------
# openpyxl 3.1.5 is installed in the `cpdb` environment, so the workbook is a
# hard requirement. The CSV bundle remains as a fallback path only; if the
# engine ever disappears the assertion below names the fix.
REQUIRE_EXCEL = True

# --- Legacy colours / labels, kept verbatim ---------------------------------
C_GOOD = "#67AD5B"   # green - enriched in Good outcome
C_POOR = "#C23F38"   # red   - enriched in Poor outcome
C_BOTH = "#4472C4"   # blue  - significant in both groups
STATUS_ORDER  = ["good_only", "both", "poor_only"]
STATUS_COLORS = {"good_only": C_GOOD, "both": C_BOTH, "poor_only": C_POOR}
STATUS_LABELS = {"good_only": "Good only", "both": "Both", "poor_only": "Poor only"}

for _d in (FIG_PRIMARY, FIG_POOLED_ALL, FIG_POOLED_GOOD, FIG_POOLED_POOR,
           FIG_POOLED_DIFF, FIG_POOLED_CONSIST, FIG_POOLED_SEEDS,
           TABLES_DIR, LOG_DIR):
    _d.mkdir(parents=True, exist_ok=True)

In [ ]:
# =============================================================================
# IMPORTS + shared-utility bindings
# =============================================================================
import json
import os
import sys
import time
import warnings
from datetime import datetime, timezone
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns
import networkx as nx
from pycirclize import Circos

warnings.filterwarnings("ignore")

plt.rcParams.update({
    "font.size": FONT_SIZE,
    "axes.titlesize": FONT_SIZE + 1,
    "axes.labelsize": FONT_SIZE,
    "xtick.labelsize": FONT_SIZE - 1,
    "ytick.labelsize": FONT_SIZE - 1,
    "legend.fontsize": FONT_SIZE - 2,
    "svg.fonttype": "none",     # editable text in Illustrator / Inkscape
    "savefig.dpi": DPI,
    "savefig.bbox": "tight",
    "pdf.fonttype": 42,
    "figure.facecolor": "white",
})

# CFG was loaded in the configuration cell; only the remaining helpers are
# imported here. Nothing in this cell re-reads or re-declares configuration.
from utils import (sha256_file, write_manifest, log_jsonl,           # noqa: E402
                   derive_patient_outcomes, safe_unlink)


def _call_flex(fn, *variants):
    last = None
    for args, kwargs in variants:
        try:
            return fn(*args, **kwargs)
        except TypeError as exc:
            last = exc
    raise last


CFG_SHA = sha256_file(CONFIG_PATH)

RUN_START = time.time()
RUN_START_ISO = datetime.now(timezone.utc).isoformat()


def _log(**record):
    rec = {"notebook": NOTEBOOK,
           "timestamp_utc": datetime.now(timezone.utc).isoformat(),
           "config_sha256": CFG_SHA, **record}
    _call_flex(log_jsonl, ((RUN_LOG, rec), {}), ((rec, RUN_LOG), {}),
               ((), {"path": RUN_LOG, "record": rec}))
    return rec


# --- figure provenance ------------------------------------------------------
FIGURE_PROVENANCE = []   # every saved figure records the table it was drawn from


def save_fig(fig, out_dir, name, source_tables, section):
    '''Write PNG + SVG into out_dir and record provenance.'''
    out_dir = Path(out_dir)
    assert str(out_dir.resolve()).startswith(str(OUTPUT_ROOT.resolve())), \
        f"refusing to write a figure outside the new output root: {out_dir}"
    out_dir.mkdir(parents=True, exist_ok=True)
    written = []
    for ext in FIG_FORMATS:
        p = out_dir / f"{name}.{ext}"
        if ext == "png":
            fig.savefig(p, dpi=DPI, bbox_inches="tight", facecolor="white")
        else:
            fig.savefig(p, format="svg", bbox_inches="tight", facecolor="white")
        written.append(p)
    plt.close(fig)
    for p in written:
        FIGURE_PROVENANCE.append({
            "section": section,
            "figure": str(p.relative_to(OUTPUT_ROOT)),
            "source_tables": ";".join(str(Path(s)) for s in np.atleast_1d(source_tables)),
        })
    return written


# --- tolerant table loading -------------------------------------------------
def find_table(patterns, roots=None, required=True):
    '''First existing file matching any glob pattern, searched under roots.'''
    roots = roots or [OUTPUT_ROOT]
    for pat in np.atleast_1d(patterns):
        for root in roots:
            hits = sorted(Path(root).glob(str(pat)))
            hits = [h for h in hits if h.is_file()]
            if hits:
                return hits[0]
    assert not required, f"no file found for patterns {list(np.atleast_1d(patterns))}"
    return None


def read_table(path):
    '''Read parquet or csv, whichever the extension says, without assuming an engine.'''
    path = Path(path)
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    if path.suffixes[-2:] == [".csv", ".gz"] or path.suffix == ".gz":
        return pd.read_csv(path, compression="gzip")
    return pd.read_csv(path)


def col(df, candidates, required=True):
    '''Resolve the first matching column name, case-insensitively.'''
    lower = {c.lower(): c for c in df.columns}
    for cand in np.atleast_1d(candidates):
        if cand in df.columns:
            return cand
        if cand.lower() in lower:
            return lower[cand.lower()]
    assert not required, f"none of {list(np.atleast_1d(candidates))} in {list(df.columns)}"
    return None

---
# Section A — Primary patient-level results

Unit of replication: the patient. Effects are rank-biserial correlations, positive
meaning larger CellPhoneDB interaction scores in the Good clinical-outcome group.
Significance is the Benjamini–Hochberg adjusted q-value computed once across the
complete estimable endpoint family in `04_`.

Nothing in this section is recomputed. Every figure is drawn from a saved table so it
can be regenerated independently, and the table it came from is recorded in
`figures/figure_provenance.csv`.

In [ ]:
# =============================================================================
# Load the finalised patient-level tables (read-only)
# =============================================================================
assert PRIMARY_RESULTS_CSV.is_file(), (
    f"{PRIMARY_RESULTS_CSV} not found - run 04_patient_level_inference.ipynb first")
primary = read_table(PRIMARY_RESULTS_CSV)

C_ID       = col(primary, ["id_cp_interaction"])
C_SENDER   = col(primary, ["sender"])
C_RECEIVER = col(primary, ["receiver"])
C_NAME     = col(primary, ["interaction_name", "interacting_pair"], required=False)
C_RB       = col(primary, ["rank_biserial"])
C_RB_LO    = col(primary, ["rank_biserial_ci_low"])
C_RB_HI    = col(primary, ["rank_biserial_ci_high"])
C_Q        = col(primary, ["q_value"])
C_P        = col(primary, ["p_value"])
C_NG       = col(primary, ["n_good"])
C_NP       = col(primary, ["n_poor"])
C_MEDG     = col(primary, ["median_good"])
C_MEDP     = col(primary, ["median_poor"])
C_TESTED   = col(primary, ["tested"], required=False)
C_AVQ      = col(primary, ["availability_q"], required=False)

tested = primary[primary[C_TESTED].astype(bool)].copy() if C_TESTED else primary.copy()
assert len(tested) > 0, "no tested endpoints in the primary results table"

# Unique keys on the tested family.
key_cols = [C_ID, C_SENDER, C_RECEIVER]
assert not tested.duplicated(key_cols).any(), "duplicate endpoint keys in primary results"

# BH must have been applied to the complete tested family exactly once.
assert tested[C_Q].notna().all(), "tested endpoints with a missing q-value"
assert (tested[C_Q] >= tested[C_P] - 1e-12).all(), "q < p somewhere: BH mis-applied"
if C_TESTED:
    untested = primary[~primary[C_TESTED].astype(bool)]
    assert untested[C_Q].isna().all() or len(untested) == 0, \
        "untested endpoints carry q-values: BH family is not the tested family"

# Evaluability rules must hold on everything that was tested.
assert (tested[C_NG] >= 6).all() and (tested[C_NP] >= 6).all(), \
    "an endpoint was tested with fewer than 6 evaluable patients in a group"

tested["neg_log10_q"] = -np.log10(tested[C_Q].clip(lower=1e-300))
tested["min_group_n"] = tested[[C_NG, C_NP]].min(axis=1)
tested["endpoint_label"] = (
    (tested[C_NAME] if C_NAME else tested[C_ID]).astype(str)
    + "  (" + tested[C_SENDER].astype(str) + " → " + tested[C_RECEIVER].astype(str) + ")"
)

n_hits = int((tested[C_Q] < FDR_ALPHA).sum())
_log(event="primary_loaded",
     source=str(PRIMARY_RESULTS_CSV),
     source_sha256=sha256_file(PRIMARY_RESULTS_CSV),
     n_rows_total=int(len(primary)), n_tested=int(len(tested)),
     n_fdr_hits=n_hits, fdr_alpha=FDR_ALPHA)

print(f"tested endpoints: {len(tested)}   endpoints with q < {FDR_ALPHA}: {n_hits}")

In [ ]:
# =============================================================================
# A1 - Participant and cell-type QC table
# =============================================================================
counts_path = find_table(["**/per_sample_celltype_counts.csv",
                          "**/*celltype_counts*.csv"], required=False)

if counts_path is not None:
    celltype_counts = read_table(counts_path)
    CC_SAMPLE = col(celltype_counts, [SAMPLE_COL, "sample", "sample_id", "patient", "patient_id"])
    CC_TYPE   = col(celltype_counts, [CELL_TYPE_COL, "cell_type", "celltype"])
    CC_N      = col(celltype_counts, ["n_cells", "count", "n"])
else:
    # Not written upstream: build the authoritative 24 x 19 matrix from the H5AD
    # obs table in backed mode (no densification) and persist it, so the figure
    # is reproducible from a saved table like every other figure here.
    import anndata as ad
    _ad = ad.read_h5ad(CFG["input_h5ad"], backed="r")
    _obs = _ad.obs[[SAMPLE_COL, CELL_TYPE_COL]].astype(str)
    _ad.file.close()
    celltype_counts = (_obs.groupby([SAMPLE_COL, CELL_TYPE_COL], observed=True)
                            .size().rename("n_cells").reset_index())
    CC_SAMPLE, CC_TYPE, CC_N = SAMPLE_COL, CELL_TYPE_COL, "n_cells"
    counts_path = TABLES_DIR / "per_sample_celltype_counts.csv"
    celltype_counts.to_csv(counts_path, index=False)

assert int(celltype_counts[CC_N].sum()) > 0

group_path = find_table(["**/patient_group_assignment.csv",
                         "**/*patient_outcome*.csv",
                         "**/*derived_patient*.csv"], required=True)
groups = read_table(group_path)
G_SAMPLE  = col(groups, ["sample_id", SAMPLE_COL, "sample", "patient", "patient_id"])
G_OUTCOME = col(groups, [OUTCOME_COL, "outcome", "group"])
outcome_map = dict(zip(groups[G_SAMPLE].astype(str), groups[G_OUTCOME].astype(str)))

assert sorted(set(outcome_map.values())) == sorted(OUTCOME_LABELS)
assert sum(v == "Good" for v in outcome_map.values()) == EXPECTED_N_GOOD
assert sum(v == "Poor" for v in outcome_map.values()) == EXPECTED_N_POOR
assert len(outcome_map) == EXPECTED_N_PATIENTS

wide = (celltype_counts
        .pivot_table(index=CC_SAMPLE, columns=CC_TYPE, values=CC_N,
                     aggfunc="sum", fill_value=0)
        .astype(int))
assert wide.shape[0] == EXPECTED_N_PATIENTS, f"expected 24 patients, got {wide.shape[0]}"
assert wide.shape[1] == EXPECTED_N_CELLTYPES, f"expected 19 cell types, got {wide.shape[1]}"

participant_qc = pd.DataFrame({
    "sample_id": wide.index.astype(str),
    "clinical_outcome": [outcome_map[str(s)] for s in wide.index],
    "n_cells_total": wide.sum(axis=1).values,
    "n_celltypes_present": (wide > 0).sum(axis=1).values,
    f"n_celltypes_retained_ge{MIN_CELLS}": (wide >= MIN_CELLS).sum(axis=1).values,
}).sort_values(["clinical_outcome", "sample_id"]).reset_index(drop=True)

celltype_qc = pd.DataFrame({
    "cell_type": wide.columns.astype(str),
    "n_cells_total": wide.sum(axis=0).values,
    "n_patients_present": (wide > 0).sum(axis=0).values,
    f"n_patients_retained_ge{MIN_CELLS}": (wide >= MIN_CELLS).sum(axis=0).values,
})
for grp in ("Good", "Poor"):
    idx = [s for s in wide.index if outcome_map[str(s)] == grp]
    celltype_qc[f"n_cells_{grp.lower()}"] = wide.loc[idx].sum(axis=0).values
    celltype_qc[f"n_patients_retained_{grp.lower()}"] = (wide.loc[idx] >= MIN_CELLS).sum(axis=0).values
celltype_qc = celltype_qc.sort_values("n_cells_total", ascending=False).reset_index(drop=True)

participant_qc.to_csv(TABLES_DIR / "table1_participant_qc.csv", index=False)
celltype_qc.to_csv(TABLES_DIR / "table2_celltype_qc.csv", index=False)

assert participant_qc["clinical_outcome"].value_counts().to_dict() == \
    {"Good": EXPECTED_N_GOOD, "Poor": EXPECTED_N_POOR}

_log(event="qc_tables",
     participant_qc=str(TABLES_DIR / "table1_participant_qc.csv"),
     celltype_qc=str(TABLES_DIR / "table2_celltype_qc.csv"),
     min_cells_threshold=MIN_CELLS)
participant_qc

In [ ]:
# =============================================================================
# A2 - Per-patient cell-type availability heatmap
# =============================================================================
order = (participant_qc.sort_values(["clinical_outcome", "sample_id"])["sample_id"].tolist())
hm = wide.reindex(index=order)
hm = hm[celltype_qc["cell_type"].tolist()]

fig, ax = plt.subplots(figsize=(max(12, hm.shape[1] * 0.75), max(8, hm.shape[0] * 0.42)))
log_counts = np.log10(hm.values + 1)
sns.heatmap(pd.DataFrame(log_counts, index=hm.index, columns=hm.columns),
            cmap="mako", ax=ax, linewidths=0.4, linecolor="white",
            cbar_kws={"label": "log10(cells + 1)"})

# Mark cells below the retention threshold - these become structural missingness.
for i in range(hm.shape[0]):
    for j in range(hm.shape[1]):
        if hm.values[i, j] < MIN_CELLS:
            ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=False,
                                       edgecolor="#C23F38", lw=1.2, hatch="///"))

ytick_colors = [C_GOOD if outcome_map[str(s)] == "Good" else C_POOR for s in hm.index]
ax.set_yticklabels(hm.index, rotation=0, fontsize=FONT_SIZE - 3)
for t, c in zip(ax.get_yticklabels(), ytick_colors):
    t.set_color(c)
    t.set_fontweight("bold")
ax.set_xticklabels(hm.columns, rotation=45, ha="right", fontsize=FONT_SIZE - 4)
ax.set_xlabel("Cell type (annot_6)")
ax.set_ylabel("Patient (green = Good, red = Poor clinical outcome)")
ax.set_title(
    "Per-patient cell-type availability\n"
    f"hatched = fewer than {MIN_CELLS} cells, excluded from that patient's CellPhoneDB run "
    "(structural missingness)",
    fontweight="bold", pad=14)
save_fig(fig, FIG_PRIMARY, "figA2_celltype_availability_heatmap",
         counts_path, "primary_patient_level")

availability_long = (hm.reset_index()
                       .melt(id_vars=hm.index.name or "index",
                             var_name="cell_type", value_name="n_cells")
                       .rename(columns={hm.index.name or "index": "sample_id"}))
availability_long["clinical_outcome"] = availability_long["sample_id"].astype(str).map(outcome_map)
availability_long["retained"] = availability_long["n_cells"] >= MIN_CELLS
availability_long.to_csv(TABLES_DIR / "table3_patient_celltype_availability.csv", index=False)
print(f"structurally unavailable patient x cell-type combinations: "
      f"{int((~availability_long['retained']).sum())} / {len(availability_long)}")

In [ ]:
# =============================================================================
# A3 - Volcano: rank-biserial effect vs -log10(q), sample-size encoded
# =============================================================================
fig, ax = plt.subplots(figsize=(10, 8))

sizes = 25 + (tested["min_group_n"] - tested["min_group_n"].min()) / \
        (tested["min_group_n"].max() - tested["min_group_n"].min() + 1e-9) * 200
sig_mask = tested[C_Q] < FDR_ALPHA

ax.scatter(tested.loc[~sig_mask, C_RB], tested.loc[~sig_mask, "neg_log10_q"],
           s=sizes[~sig_mask], c="#B8B8B8", alpha=0.55,
           edgecolors="white", linewidths=0.3, zorder=2, label="q ≥ 0.05")
if sig_mask.any():
    sub = tested[sig_mask]
    cols = np.where(sub[C_RB] > 0, C_GOOD, C_POOR)
    ax.scatter(sub[C_RB], sub["neg_log10_q"], s=sizes[sig_mask], c=cols,
               edgecolors="#333333", linewidths=0.5, zorder=3,
               label=f"q < {FDR_ALPHA}")

ax.axhline(-np.log10(FDR_ALPHA), color="#444444", ls="--", lw=1.0, zorder=1)
ax.axvline(0, color="#444444", lw=0.8, zorder=1)
ax.text(ax.get_xlim()[1], -np.log10(FDR_ALPHA), f"  q = {FDR_ALPHA}",
        va="bottom", ha="right", fontsize=FONT_SIZE - 3, color="#444444")
ax.set_xlabel("Rank-biserial correlation  (positive = higher scores in Good outcome group)")
ax.set_ylabel("−log₁₀(Benjamini–Hochberg q-value)")
ax.set_title(
    "Patient-level comparison of CellPhoneDB interaction scores by clinical-outcome group\n"
    f"{len(tested):,} estimable endpoints (interaction × sender × receiver); "
    "point size = smaller group n",
    fontweight="bold", pad=12)

size_handles = [
    Line2D([0], [0], marker="o", linestyle="None", markerfacecolor="#888888",
           markeredgecolor="white", markersize=ms, label=f"min(n_good, n_poor) = {v}")
    for ms, v in zip([6, 9, 13],
                     [int(tested["min_group_n"].min()),
                      int(tested["min_group_n"].median()),
                      int(tested["min_group_n"].max())])
]
leg1 = ax.legend(handles=size_handles, loc="upper left", frameon=True,
                 framealpha=0.92, edgecolor="#cccccc", fontsize=FONT_SIZE - 4)
ax.add_artist(leg1)
ax.legend(handles=[mpatches.Patch(color=C_GOOD, label="q < 0.05, higher in Good"),
                   mpatches.Patch(color=C_POOR, label="q < 0.05, higher in Poor"),
                   mpatches.Patch(color="#B8B8B8", label="q ≥ 0.05")],
          loc="upper right", frameon=True, framealpha=0.92,
          edgecolor="#cccccc", fontsize=FONT_SIZE - 4)
sns.despine(ax=ax)
save_fig(fig, FIG_PRIMARY, "figA3_volcano_rank_biserial",
         PRIMARY_RESULTS_CSV, "primary_patient_level")

In [ ]:
# =============================================================================
# A4 - Forest plot: effect sizes with 95% bootstrap CIs
#      FDR hits if any; otherwise the top 20 by q-value, labelled as an
#      exploratory ranking (NOT a result set).
# =============================================================================
hits = tested[tested[C_Q] < FDR_ALPHA].sort_values(C_Q)
if len(hits) > 0:
    forest = hits.copy()
    forest_mode = "fdr_hits"
    forest_title = (f"Endpoints associated with clinical-outcome group  "
                    f"(Benjamini–Hochberg q < {FDR_ALPHA}; n = {len(forest)})")
    forest_note = "Rank-biserial correlation with 95% stratified patient bootstrap CI."
else:
    forest = tested.nsmallest(N_FOREST, C_Q).copy()
    forest_mode = "exploratory_ranking_no_fdr_hits"
    forest_title = (f"EXPLORATORY RANKING — top {len(forest)} endpoints by q-value\n"
                    f"NO endpoint reached Benjamini–Hochberg q < {FDR_ALPHA}")
    forest_note = ("No endpoint met the prespecified q < 0.05 threshold. These are the "
                   "smallest q-values in the family, shown as an exploratory ranking only; "
                   "they are not statistically significant findings.")

forest = forest.sort_values(C_RB).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(11, max(5, len(forest) * 0.42 + 2.2)))
y = np.arange(len(forest))
colors = np.where(forest[C_RB] > 0, C_GOOD, C_POOR)

for i, r in forest.iterrows():
    ax.plot([r[C_RB_LO], r[C_RB_HI]], [i, i], color=colors[i], lw=2.0,
            alpha=0.85, solid_capstyle="butt", zorder=2)
    ax.plot([r[C_RB_LO]] * 2, [i - 0.16, i + 0.16], color=colors[i], lw=1.4, zorder=2)
    ax.plot([r[C_RB_HI]] * 2, [i - 0.16, i + 0.16], color=colors[i], lw=1.4, zorder=2)
ax.scatter(forest[C_RB], y, s=70, c=colors, edgecolors="white", linewidths=0.9, zorder=3)

ax.axvline(0, color="#444444", lw=1.0, ls="--", zorder=1)
ax.set_yticks(y)
ax.set_yticklabels(forest["endpoint_label"], fontsize=FONT_SIZE - 4)
ax.set_ylim(-0.7, len(forest) - 0.3)
ax.set_xlabel("Rank-biserial correlation (Good − Poor), 95% bootstrap CI")
ax.set_title(forest_title, fontweight="bold", pad=12)
ax.text(0.5, -0.10 if len(forest) > 8 else -0.18, forest_note, transform=ax.transAxes,
        ha="center", va="top", fontsize=FONT_SIZE - 4, color="#555555", style="italic")
ax.legend(handles=[mpatches.Patch(color=C_GOOD, label="higher in Good outcome group"),
                   mpatches.Patch(color=C_POOR, label="higher in Poor outcome group")],
          loc="lower right", frameon=True, framealpha=0.92,
          edgecolor="#cccccc", fontsize=FONT_SIZE - 4)
sns.despine(ax=ax, left=True)
save_fig(fig, FIG_PRIMARY, "figA4_forest_effect_sizes",
         PRIMARY_RESULTS_CSV, "primary_patient_level")

forest_out = forest.copy()
forest_out["forest_mode"] = forest_mode
forest_out.to_csv(TABLES_DIR / "table4_forest_endpoints.csv", index=False)
_log(event="forest_plot", mode=forest_mode, n_endpoints=int(len(forest)),
     n_fdr_hits=int(len(hits)))
print(f"forest mode: {forest_mode}   endpoints plotted: {len(forest)}")

In [ ]:
# =============================================================================
# A5 - Patient-level jitter / box plots for selected endpoints.
#      Every patient is drawn; structural missingness is counted, never zeroed.
# =============================================================================
long_path = find_table(["**/patient_endpoint_scores_long.parquet",
                        "**/patient_endpoint_scores_long.csv*",
                        "**/patient_endpoint_scores*.parquet",
                        "**/patient_endpoint_scores*.csv*"], required=True)
long_df = read_table(long_path)

L_ID     = col(long_df, ["id_cp_interaction"])
L_SEND   = col(long_df, ["sender"])
L_RECV   = col(long_df, ["receiver"])
L_SAMPLE = col(long_df, ["sample_id", SAMPLE_COL, "sample", "patient", "patient_id"])
L_SCORE  = col(long_df, ["interaction_score", "score", "value"])
L_STATUS = col(long_df, ["availability", "status", "endpoint_status"], required=False)

long_df[L_SAMPLE] = long_df[L_SAMPLE].astype(str)
long_df["clinical_outcome"] = long_df[L_SAMPLE].map(outcome_map)
assert long_df["clinical_outcome"].notna().all(), \
    "a patient in the long score table has no clinical_outcome label"

selected = forest.head(N_JITTER_ENDPOINTS).sort_values(C_Q)
n_panels = len(selected)
ncols = min(3, n_panels)
nrows = int(np.ceil(n_panels / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 4.8 * nrows), squeeze=False)

jitter_rows = []
rng = np.random.default_rng(0)
for k, (_, ep) in enumerate(selected.iterrows()):
    ax = axes[k // ncols][k % ncols]
    sub = long_df[(long_df[L_ID] == ep[C_ID]) &
                  (long_df[L_SEND] == ep[C_SENDER]) &
                  (long_df[L_RECV] == ep[C_RECEIVER])]
    n_missing = EXPECTED_N_PATIENTS - int(sub[L_SCORE].notna().sum())
    data, positions = [], []
    for gi, grp in enumerate(("Good", "Poor")):
        vals = sub.loc[sub["clinical_outcome"] == grp, L_SCORE].dropna().values.astype(float)
        data.append(vals)
        positions.append(gi)
        x = np.full(len(vals), gi, dtype=float) + rng.uniform(-0.11, 0.11, len(vals))
        ax.scatter(x, vals, s=52, c=(C_GOOD if grp == "Good" else C_POOR),
                   edgecolors="#333333", linewidths=0.5, alpha=0.9, zorder=3)
        jitter_rows.append({"id_cp_interaction": ep[C_ID], "sender": ep[C_SENDER],
                            "receiver": ep[C_RECEIVER], "clinical_outcome": grp,
                            "n_evaluable": int(len(vals)),
                            "n_expected": EXPECTED_N_GOOD if grp == "Good" else EXPECTED_N_POOR})
    bp = ax.boxplot(data, positions=positions, widths=0.5, showfliers=False,
                    patch_artist=True, zorder=2)
    for patch in bp["boxes"]:
        patch.set_facecolor("#FFFFFF")
        patch.set_edgecolor("#555555")
    for med in bp["medians"]:
        med.set_color("#222222")
        med.set_linewidth(1.6)

    ax.set_xticks([0, 1])
    ax.set_xticklabels([f"Good\nn={len(data[0])}", f"Poor\nn={len(data[1])}"])
    ax.set_ylabel("CellPhoneDB interaction score")
    ax.set_title(str(ep["endpoint_label"])[:70], fontsize=FONT_SIZE - 4, fontweight="bold")
    ax.text(0.02, 0.98,
            f"r$_{{rb}}$ = {ep[C_RB]:.2f}  [{ep[C_RB_LO]:.2f}, {ep[C_RB_HI]:.2f}]\n"
            f"q = {ep[C_Q]:.3g}\nstructurally missing patients: {n_missing}",
            transform=ax.transAxes, va="top", ha="left",
            fontsize=FONT_SIZE - 5, color="#444444")
    sns.despine(ax=ax)

for k in range(n_panels, nrows * ncols):
    axes[k // ncols][k % ncols].axis("off")

fig.suptitle("Patient-level interaction scores by clinical-outcome group\n"
             "every evaluable patient shown; structural missingness counted, never imputed as zero",
             fontsize=FONT_SIZE + 1, fontweight="bold", y=1.0)
fig.tight_layout()
save_fig(fig, FIG_PRIMARY, "figA5_patient_jitter_selected_endpoints",
         [long_path, PRIMARY_RESULTS_CSV], "primary_patient_level")

pd.DataFrame(jitter_rows).to_csv(TABLES_DIR / "table5_jitter_endpoint_counts.csv", index=False)

In [ ]:
# =============================================================================
# A6 - Availability-bias flags
#      Endpoint availability that differs by outcome makes the conditional score
#      comparison selection-sensitive. Flagged, not silently carried.
# =============================================================================
if C_AVQ:
    flags = tested[[C_ID, C_SENDER, C_RECEIVER, C_NG, C_NP, C_RB, C_Q, C_AVQ]].copy()
    flags["availability_bias_flag"] = flags[C_AVQ] < FDR_ALPHA
    n_flagged = int(flags["availability_bias_flag"].sum())
else:
    flags = tested[[C_ID, C_SENDER, C_RECEIVER, C_NG, C_NP, C_RB, C_Q]].copy()
    flags["availability_bias_flag"] = pd.NA
    n_flagged = 0

flags["n_structurally_missing"] = EXPECTED_N_PATIENTS - (flags[C_NG] + flags[C_NP])
flags.to_csv(TABLES_DIR / "table6_availability_bias_flags.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 6))
ax.hist(flags["n_structurally_missing"], bins=np.arange(-0.5, EXPECTED_N_PATIENTS + 1.5, 1),
        color="#6A8CAF", edgecolor="white")
ax.set_xlabel("Patients not evaluable for the endpoint (structural missingness)")
ax.set_ylabel("Number of endpoints")
ax.set_title("Structural missingness across the tested endpoint family\n"
             f"{n_flagged} endpoints additionally carry an availability-bias flag "
             f"(availability q < {FDR_ALPHA})",
             fontweight="bold", pad=12)
sns.despine(ax=ax)
save_fig(fig, FIG_PRIMARY, "figA6_structural_missingness",
         PRIMARY_RESULTS_CSV, "primary_patient_level")

_log(event="availability_flags", n_flagged=n_flagged,
     availability_column_present=bool(C_AVQ))
print(f"endpoints flagged for availability bias: {n_flagged} / {len(flags)}")

In [ ]:
# =============================================================================
# A7 - Primary results CSV + workbook with a data dictionary
# =============================================================================
primary_out = TABLES_DIR / "primary_results.csv"
primary.to_csv(primary_out, index=False)

DICTIONARY = [
    ("id_cp_interaction", "CellPhoneDB ligand-receptor interaction identifier."),
    ("sender",   "Cell type expressing the ligand side (annot_6). Direction is preserved."),
    ("receiver", "Cell type expressing the receptor side (annot_6)."),
    ("interaction_name", "Human-readable ligand-receptor pair."),
    ("n_good", "Evaluable Good-outcome patients (both cell types retained and a score computed)."),
    ("n_poor", "Evaluable Poor-outcome patients."),
    ("nonzero_good", "Evaluable Good patients with a non-zero score."),
    ("nonzero_poor", "Evaluable Poor patients with a non-zero score."),
    ("median_good", "Median CellPhoneDB interaction score among evaluable Good patients."),
    ("median_poor", "Median CellPhoneDB interaction score among evaluable Poor patients."),
    ("median_difference", "median_good minus median_poor."),
    ("rank_biserial", "Rank-biserial correlation, 2*U/(n_good*n_poor) - 1. "
                      "Positive = larger scores in the Good clinical-outcome group."),
    ("rank_biserial_ci_low",  "Lower bound, 95% stratified patient bootstrap percentile CI."),
    ("rank_biserial_ci_high", "Upper bound, 95% stratified patient bootstrap percentile CI."),
    ("median_difference_ci_low",  "Lower bound, 95% bootstrap CI for the median difference."),
    ("median_difference_ci_high", "Upper bound, 95% bootstrap CI for the median difference."),
    ("mannwhitney_u", "Mann-Whitney U statistic, two-sided."),
    ("p_value", "Two-sided Mann-Whitney p-value: exact when the endpoint is tie-free, "
                "asymptotic with tie correction otherwise."),
    ("q_value", "Benjamini-Hochberg adjusted p-value, computed once across the complete "
                "estimable primary endpoint family."),
    ("availability_odds_ratio", "Fisher odds ratio for endpoint evaluability by outcome group."),
    ("availability_p", "Fisher exact p-value for differential evaluability."),
    ("availability_q", "Benjamini-Hochberg q-value within the separate availability family."),
    ("permutation_p", "Patient-label permutation p-value, (extreme + 1)/(valid + 1). "
                      "Prespecified sensitivity check, not a second discovery filter."),
    ("permutation_valid_n", "Permutations in which the endpoint remained estimable."),
    ("tested", "True if the endpoint met the 6/6 evaluability and prevalence rules and "
               "entered the BH family."),
    ("exclusion_reason", "Why an endpoint was not tested."),
    ("analysis_scope", "primary_patient_level. Exploratory pooled tables are stored "
                       "separately under exploratory_pooled/."),
]
dict_df = pd.DataFrame(
    [(c, d) for c, d in DICTIONARY if c in primary.columns or c == "analysis_scope"],
    columns=["column", "description"])
dict_df.to_csv(TABLES_DIR / "primary_results_data_dictionary.csv", index=False)

SHEETS = {
    "primary_results": primary,
    "data_dictionary": dict_df,
    "participant_qc": participant_qc,
    "celltype_qc": celltype_qc,
    "availability_flags": flags,
    "forest_endpoints": forest_out,
}

excel_engine = None
for _eng in ("openpyxl", "xlsxwriter"):
    try:
        __import__(_eng)
        excel_engine = _eng
        break
    except ImportError:
        continue

workbook_path = TABLES_DIR / "primary_results_workbook.xlsx"
if excel_engine:
    with pd.ExcelWriter(workbook_path, engine=excel_engine) as xl:
        for sheet, df in SHEETS.items():
            df.to_excel(xl, sheet_name=sheet[:31], index=False)
    workbook_status = f"xlsx:{excel_engine}"
else:
    assert not REQUIRE_EXCEL, (
        "no Excel engine in the `cpdb` environment. Install one with:\n"
        "  conda run -n cpdb pip install openpyxl\n"
        "or set REQUIRE_EXCEL = False to accept the CSV bundle fallback.")
    bundle = TABLES_DIR / "primary_results_workbook_csv_bundle"
    bundle.mkdir(parents=True, exist_ok=True)
    for sheet, df in SHEETS.items():
        df.to_csv(bundle / f"{sheet}.csv", index=False)
    workbook_path = bundle
    workbook_status = "csv_bundle_fallback_no_excel_engine"

_log(event="primary_tables_written", primary_csv=str(primary_out),
     workbook=str(workbook_path), workbook_status=workbook_status,
     n_dictionary_entries=int(len(dict_df)))
print(f"workbook: {workbook_status} -> {workbook_path}")

---
# Section B — Exploratory pooled-cell figures

Everything below pools cells across patients within an outcome group. The unit of
replication is the cell-type pair or the ligand–receptor interaction, **not the
patient**, so nothing here supports a patient-level clinical claim. Each figure carries
the subtitle *"Exploratory pooled-cell analysis"*.

### Interaction coverage governs every count plotted below

The expression matrix is a 10,000-highly-variable-gene subset, so only **1,224 of the
2,911** CellPhoneDB database interactions have both partners present and are evaluable
at all (`audit/cpdb_interaction_coverage.json`). Every interaction count, heatmap cell,
chord width and pathway bar in this section is therefore a count **within that evaluable
subset**, never within the database.

Coverage is **not uniform** across pathways — it ranges from 0 % to 100 %, so the
aggregate figure hides what matters. Three consequences for reading these figures:

* **A pathway bar that is short, or a heatmap cell that is empty, is not evidence of
  biological absence.** It may simply mean one partner's gene did not survive gene
  selection. 22 classifications have *zero* detectable interactions, so for those a
  count of zero is guaranteed by arithmetic and carries no information at all.
* **`Signaling by WNT` is covered 2/355 (0.6 %) while `Signaling by WNT inhibition` is
  covered 113/123 (91.9 %).** Wherever both appear, WNT signalling will look near-absent
  and WNT inhibition prominent. **That contrast is a coverage artifact of gene selection,
  not biology**, and must not be read as WNT suppression in ACC. It is the clearest
  example of how this gap can produce a confident but false narrative. (This is purely
  about how many interactions survived gene selection — it is not a pathway-matching
  problem, and `WNT` is not one of the `ACC_PATHWAYS` keys, so figures 06 and 08 are not
  affected.)
* **`Signaling by Notch` is covered 13/26 (50 %)**, which matters because a dedicated
  Notch figure is produced below. A positive Notch finding remains a real observation,
  but the figure cannot support any statement about the *extent* of Notch signalling in
  ACC, and a negative one would be worthless.

Comparisons *between outcome groups* remain valid, because both groups are measured on
the same restricted gene set. Comparisons *between pathways* are not, unless their
coverage is comparable. `LIMITATIONS.md` gives the full per-classification table.

**Pathway selection was checked and is sound.** The ported figures select pathways with
`classification.str.contains(key, na=False)`, which is case-sensitive. All eight
`ACC_PATHWAYS` keys were verified against the current database to match **exactly one**
classification each — Notch, Fibroblast growth factor, Transforming growth factor,
Vascular endothelial, Epidermal growth factor, Ephrin, Tumor necrosis, Chemokine. There
is no conflation between pathways. In particular `'Ephrin'` does not match
`'Signaling by Epinephrine'`, because the substring in that name is `'nephrine'` and the
match is case-sensitive. The driver cell below re-asserts this at runtime, so a future
database that introduces a genuine collision fails loudly rather than silently merging
two pathways into one bar.

## B1 — Descriptive CellPhoneDB maps (`cpdb_plots_generation.py`)

The canonical 805-line argparse script is ported below. Its `--base-dir` argument is
replaced by the config cell, and the module-level globals (`sig_means`, `pvals`,
`RESULTS_DIR`, …) become fields on an explicit context object so the same code can be
driven three times — **all samples, pooled Good, pooled Poor** — without hidden state.
The plotting logic, the label maps (`SHORT_NAMES`, `CELL_ORDER`, `CELL_CATEGORIES`,
`CELL_COLORS_MUTED`, `ACC_PATHWAYS`) and the figure filenames are unchanged.

Running it for pooled Good as well as pooled Poor fixes the old asymmetry where
descriptive plots existed only for the Poor group.

In [ ]:
# =============================================================================
# PORTED from legacy_sources/cpdb_plots_generation.py (canonical argparse version)
# Label maps verbatim. Verified 19/19 coverage of annot_6 for SHORT_NAMES,
# CELL_CATEGORIES and CELL_ORDER.
# =============================================================================

SHORT_NAMES = {
    'ADC - Tumor': 'ADC - Tumor',
    'ADMEC - Tumor': 'ADMEC - Tumor',
    'B cells': 'B cells',
    'Basal cells - Tumor': 'Basal - Tumor',
    'CD4+ T cells': 'CD4+ T',
    'CD8+ T cells': 'CD8+ T',
    'Endothelial cells': 'Endothelial',
    'Epithelial cells': 'Epithelial',
    'Epithelial cells - Tumor': 'Epithelial - Tumor',
    'Fibroblast cells': 'Fibroblast',
    'Fibroblast cells - CAF': 'Fibroblast - CAF',
    'Fibroblast cells - Inflammatory': 'Fibroblast - Inflam',
    'Macrophages - AP/TAM': 'Macrophage - TAM',
    'Macrophages - M2': 'Macrophage M2',
    'Mast cells': 'Mast',
    'Memory B cells': 'Memory B',
    'Mural cells': 'Mural',
    'Muscle cells': 'Muscle',
    'Plasma cells': 'Plasma',
}

CELL_ORDER = [
    'Epithelial cells', 'Epithelial cells - Tumor', 'Basal cells - Tumor',
    'ADC - Tumor', 'ADMEC - Tumor',
    'Fibroblast cells', 'Fibroblast cells - CAF', 'Fibroblast cells - Inflammatory',
    'Endothelial cells', 'Mural cells', 'Muscle cells',
    'CD4+ T cells', 'CD8+ T cells', 'Macrophages - AP/TAM', 'Macrophages - M2',
    'B cells', 'Memory B cells', 'Plasma cells', 'Mast cells',
]

CELL_CATEGORIES = {
    'Epithelial cells': 'Tumor', 'Epithelial cells - Tumor': 'Tumor',
    'Basal cells - Tumor': 'Tumor', 'ADC - Tumor': 'Tumor', 'ADMEC - Tumor': 'Tumor',
    'Fibroblast cells': 'Stroma', 'Fibroblast cells - CAF': 'Stroma',
    'Fibroblast cells - Inflammatory': 'Stroma', 'Endothelial cells': 'Stroma',
    'Mural cells': 'Stroma', 'Muscle cells': 'Stroma',
    'CD4+ T cells': 'Immune', 'CD8+ T cells': 'Immune',
    'Macrophages - AP/TAM': 'Immune', 'Macrophages - M2': 'Immune',
    'B cells': 'Immune', 'Memory B cells': 'Immune', 'Plasma cells': 'Immune',
    'Mast cells': 'Immune',
}

CELL_COLORS_MUTED = {
    'Epithelial cells': '#A6CEE3', 'Epithelial cells - Tumor': '#1F78B4',
    'Basal cells - Tumor': '#B2DF8A', 'ADC - Tumor': '#FB9A99',
    'ADMEC - Tumor': '#FFED6F', 'Fibroblast cells': '#FDBF6F',
    'Fibroblast cells - CAF': '#FF7F00', 'Fibroblast cells - Inflammatory': '#F4A460',
    'Endothelial cells': '#CAB2D6', 'Mural cells': '#D9D9D9', 'Muscle cells': '#E5D8BD',
    'CD4+ T cells': '#B3CDE3', 'CD8+ T cells': '#8DD3C7',
    'Macrophages - AP/TAM': '#CCEBC5', 'Macrophages - M2': '#80B1D3',
    'B cells': '#DECBE4', 'Memory B cells': '#BC80BD', 'Plasma cells': '#FED9A6',
    'Mast cells': '#FBB4AE',
}

ACC_PATHWAYS = {
    'Notch': 'Notch',
    'Fibroblast growth factor': 'FGF',
    'Transforming growth factor': 'TGF-β',
    'Vascular endothelial': 'VEGF',
    'Epidermal growth factor': 'EGF',
    'Ephrin': 'Ephrin',
    'Tumor necrosis': 'TNF',
    'Chemokine': 'Chemokine',
}

assert len(SHORT_NAMES) == len(CELL_CATEGORIES) == len(CELL_ORDER) == EXPECTED_N_CELLTYPES
assert set(SHORT_NAMES) == set(CELL_CATEGORIES) == set(CELL_ORDER), \
    "label maps disagree on the annot_6 label set"


def shorten(name):
    return SHORT_NAMES.get(name, name)


def make_ctx(base_dir, out_dir, cohort_label, subtitle):
    '''Build the plotting context that replaces the legacy module-level globals.'''
    base_dir, out_dir = Path(base_dir), Path(out_dir)
    for f in ('means.csv', 'pvalues.csv', 'significant_means.csv', 'interaction_scores.csv'):
        assert (base_dir / f).is_file(), f"missing {base_dir / f}"
    out_dir.mkdir(parents=True, exist_ok=True)

    sig_means = pd.read_csv(base_dir / 'significant_means.csv')
    cell_pairs = [c for c in sig_means.columns if '|' in c]
    cell_types = sorted(set(ct for cp in cell_pairs for ct in cp.split('|')))
    unmapped = [c for c in cell_types if c not in SHORT_NAMES]
    assert not unmapped, f"cell types present in {base_dir} but absent from the label maps: {unmapped}"

    order = [c for c in CELL_ORDER if c in cell_types]
    return SimpleNamespace(
        base_dir=base_dir, out_dir=out_dir,
        means=pd.read_csv(base_dir / 'means.csv'),
        pvals=pd.read_csv(base_dir / 'pvalues.csv'),
        sig_means=sig_means,
        scores=pd.read_csv(base_dir / 'interaction_scores.csv'),
        cell_pairs=cell_pairs, cell_types=cell_types, order=order,
        cohort_label=cohort_label, subtitle=subtitle,
        ttl=lambda s, _sub=subtitle: f"{s}\n{_sub}",
    )


def build_sig_count_matrix(df, pairs, order):
    '''Count number of significant interactions per cell-type pair.'''
    mat = pd.DataFrame(0, index=order, columns=order)
    for cp in pairs:
        a, b = cp.split('|')
        if a in order and b in order:
            mat.loc[a, b] = df[cp].notna().sum()
    return mat


def build_mean_matrix(df, pairs, order):
    '''Matrix of mean values across all interactions for each cell-pair.'''
    mat = pd.DataFrame(0.0, index=order, columns=order)
    for cp in pairs:
        a, b = cp.split('|')
        if a in order and b in order:
            vals = df[cp].dropna()
            if len(vals) > 0:
                mat.loc[a, b] = vals.mean()
    return mat

In [ ]:
# =============================================================================
# cpdb_plots_generation.py - plot functions 1-9, bodies unchanged apart from
# taking `ctx` instead of module globals and routing titles through ctx.ttl().
# =============================================================================

def plot_significant_interaction_heatmap(ctx):
    mat = build_sig_count_matrix(ctx.sig_means, ctx.cell_pairs, ctx.order)
    mat_short = mat.rename(index=shorten, columns=shorten)

    fig, ax = plt.subplots(figsize=(13, 11))
    mask = mat_short.values == 0
    sns.heatmap(mat_short, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.5,
                linecolor='white', ax=ax, mask=mask,
                cbar_kws={'label': 'Number of Significant Interactions'},
                square=True, vmin=0, annot_kws={'size': 10})
    ax.set_title(ctx.ttl('Significant Cell-Cell Interactions in ACC\n'
                         f'(CellPhoneDB, p < 0.05, {ctx.cohort_label})'),
                 fontsize=FONT_SIZE, fontweight='bold', pad=15)
    ax.set_xlabel('Receiver Cell Type', fontsize=FONT_SIZE)
    ax.set_ylabel('Sender Cell Type', fontsize=FONT_SIZE)
    plt.xticks(rotation=45, ha='right', fontsize=FONT_SIZE - 2)
    plt.yticks(rotation=0, fontsize=FONT_SIZE - 2)
    return fig, '01_significant_interaction_heatmap'


def plot_mean_interaction_heatmap(ctx):
    mat = build_mean_matrix(ctx.sig_means, ctx.cell_pairs, ctx.order)
    mat_short = mat.rename(index=shorten, columns=shorten)

    fig, ax = plt.subplots(figsize=(13, 11))
    mask = mat_short.values == 0
    sns.heatmap(mat_short, annot=True, fmt='.2f', cmap='viridis', linewidths=0.5,
                linecolor='white', ax=ax, mask=mask,
                cbar_kws={'label': 'Mean Expression (Significant Interactions)'},
                square=True, annot_kws={'size': 10})
    ax.set_title(ctx.ttl('Mean Interaction Strength in ACC Tumor Microenvironment\n'
                         '(Significant interactions only)'),
                 fontsize=FONT_SIZE, fontweight='bold', pad=15)
    ax.set_xlabel('Receiver Cell Type', fontsize=FONT_SIZE)
    ax.set_ylabel('Sender Cell Type', fontsize=FONT_SIZE)
    plt.xticks(rotation=45, ha='right', fontsize=FONT_SIZE - 2)
    plt.yticks(rotation=0, fontsize=FONT_SIZE - 2)
    return fig, '02_mean_interaction_strength_heatmap'


def _make_chord(mat_df, title, label_size=FONT_SIZE - 2):
    '''Single chord diagram with no tick numbers. Returns None if degenerate.'''
    nonzero = mat_df.loc[(mat_df.sum(axis=1) > 0), (mat_df.sum(axis=0) > 0)]
    if nonzero.shape[0] < 2:
        return None

    circos = Circos.initialize_from_matrix(
        nonzero, space=4, cmap="Set2",
        label_kws=dict(size=label_size, r=110),
        link_kws=dict(direction=1, ec="black", lw=0.2),
    )
    fig = circos.plotfig(figsize=(14, 14))
    fig.suptitle(title, fontsize=FONT_SIZE, fontweight='bold', y=0.98)
    return fig


def plot_chord_diagrams(ctx):
    mat = build_sig_count_matrix(ctx.sig_means, ctx.cell_pairs, ctx.order)
    labels = [shorten(ct) for ct in ctx.order]
    mat_df = pd.DataFrame(mat.values.astype(float), index=labels, columns=labels)

    out = []
    out.append((_make_chord(mat_df, ctx.ttl(
        'Cell-Cell Communication Network in ACC\n'
        '(Chord width = number of significant interactions)')),
        '03a_chord_diagram_all'))

    for cat, fname, desc in [
        ('Tumor',  '03b_chord_diagram_tumor',  'Tumor Compartment Interactions'),
        ('Stroma', '03c_chord_diagram_stroma', 'Stromal Compartment Interactions'),
        ('Immune', '03d_chord_diagram_immune', 'Immune Compartment Interactions'),
    ]:
        types = [shorten(ct) for ct in ctx.order if CELL_CATEGORIES[ct] == cat]
        if len(types) < 2:
            out.append((None, fname))
            continue
        out.append((_make_chord(mat_df.loc[types, types], ctx.ttl(desc)), fname))
    return out


def plot_network_diagram(ctx):
    mat = build_sig_count_matrix(ctx.sig_means, ctx.cell_pairs, ctx.order)

    G = nx.DiGraph()
    for ct in ctx.order:
        G.add_node(shorten(ct), category=CELL_CATEGORIES[ct])
    for i, ct_a in enumerate(ctx.order):
        for j, ct_b in enumerate(ctx.order):
            w = mat.iloc[i, j]
            if w > 0 and ct_a != ct_b:
                G.add_edge(shorten(ct_a), shorten(ct_b), weight=w)

    pos = nx.spring_layout(G, k=3.5, iterations=300, seed=42)
    fig, ax = plt.subplots(figsize=(18, 16))

    node_colors = [CELL_COLORS_MUTED[ct] for ct in ctx.order]
    node_sizes = [400 + (mat.loc[ct].sum() + mat[ct].sum()) * 8 for ct in ctx.order]

    edges = list(G.edges(data=True))
    edge_weights = [d['weight'] for _, _, d in edges]
    max_w = max(edge_weights) if edge_weights else 1
    edge_widths = [0.5 + (w / max_w) * 4 for w in edge_weights]
    edge_alphas = [0.15 + (w / max_w) * 0.55 for w in edge_weights]

    for (u, v, d), width, alpha in zip(edges, edge_widths, edge_alphas):
        nx.draw_networkx_edges(G, pos, edgelist=[(u, v)], width=width,
                               alpha=alpha, edge_color='#666666',
                               arrows=True, arrowsize=14,
                               connectionstyle='arc3,rad=0.1', ax=ax)
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes,
                           edgecolors='#333333', linewidths=1.0, ax=ax)

    pos_arr = np.array(list(pos.values()))
    center = pos_arr.mean(axis=0)
    label_gap = 0.18

    for node, (x, y) in pos.items():
        dx, dy = x - center[0], y - center[1]
        dist = np.hypot(dx, dy)
        ux, uy = (dx / dist, dy / dist) if dist > 0 else (1.0, 0.0)
        lx, ly = x + ux * label_gap, y + uy * label_gap
        ha = 'left' if ux >= 0 else 'right'
        va = 'bottom' if uy >= 0.3 else ('top' if uy <= -0.3 else 'center')
        ax.annotate(node, xy=(x, y), xytext=(lx, ly),
                    fontsize=FONT_SIZE - 3, fontweight='bold', ha=ha, va=va,
                    arrowprops=dict(arrowstyle='-', color='#888888',
                                    lw=0.7, shrinkA=6, shrinkB=2),
                    bbox=dict(boxstyle='round,pad=0.15', facecolor='white',
                              edgecolor='none', alpha=0.75))

    legend_elements = [
        mpatches.Patch(facecolor='#A6CEE3', edgecolor='#333333',
                       label='Tumor (Epithelial / Epithelial-T / Basal-T / ADC-T / ADMEC-T)'),
        mpatches.Patch(facecolor='#FDBF6F', edgecolor='#333333',
                       label='Stroma (Fibro. / CAF / Inflam.Fibro. / Endo. / Mural / Muscle)'),
        mpatches.Patch(facecolor='#CCEBC5', edgecolor='#333333',
                       label='Immune (CD4+T / CD8+T / TAM / M2 / B / Mem.B / Plasma / Mast)'),
    ]
    ax.legend(handles=legend_elements, loc='upper center',
              bbox_to_anchor=(0.5, -0.01), ncol=3,
              fontsize=FONT_SIZE - 2, frameon=True, fancybox=True, framealpha=0.9)
    ax.set_title(ctx.ttl('Cell-Cell Communication Network in ACC TME\n'
                         '(Node size = total interactions, Edge width = interaction count)'),
                 fontsize=FONT_SIZE, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    return fig, '04_network_diagram_cell_communication'


def plot_notch_pathway_heatmap(ctx):
    notch = ctx.sig_means[ctx.sig_means['classification'].str.contains('Notch', na=False)]
    if len(notch) == 0:
        return None, '05_notch_signaling_heatmap'

    active_pairs = [cp for cp in ctx.cell_pairs if notch[cp].notna().any()]
    if len(active_pairs) == 0:
        return None, '05_notch_signaling_heatmap'
    plot_data = notch.set_index('interacting_pair')[active_pairs].copy()
    plot_data.columns = [c.replace('|', ' → ') for c in plot_data.columns]

    def shorten_pair(s):
        parts = s.split(' → ')
        return f"{shorten(parts[0])} → {shorten(parts[1])}"
    plot_data.columns = [shorten_pair(c) for c in plot_data.columns]

    notch_pvals = ctx.pvals[ctx.pvals['interacting_pair'].isin(notch['interacting_pair'])]
    pval_data = notch_pvals.set_index('interacting_pair')[active_pairs].copy()
    pval_data.columns = plot_data.columns

    fig, ax = plt.subplots(figsize=(max(18, len(active_pairs) * 0.65), max(7, len(notch) * 0.6)))
    sns.heatmap(plot_data.astype(float), cmap='RdYlBu_r', linewidths=0.5, linecolor='white',
                ax=ax, cbar_kws={'label': 'Mean Expression'},
                xticklabels=True, yticklabels=True)

    for i, row_name in enumerate(plot_data.index):
        for j, col_name in enumerate(plot_data.columns):
            val = plot_data.iloc[i, j]
            pval = pval_data.loc[row_name].iloc[j] if row_name in pval_data.index else 1.0
            if pd.notna(val) and pd.notna(pval) and pval < 0.05:
                ax.plot(j + 0.5, i + 0.5, 'k*', markersize=6)

    ax.set_title(ctx.ttl('Notch Signaling in ACC Tumor Microenvironment\n'
                         '(Key driver of ACC via MYB-NFIB fusion; * = p < 0.05)'),
                 fontsize=FONT_SIZE, fontweight='bold', pad=15)
    ax.set_xlabel('Cell-Cell Pair (Sender → Receiver)', fontsize=FONT_SIZE)
    ax.set_ylabel('Ligand-Receptor Pair', fontsize=FONT_SIZE)
    plt.xticks(rotation=45, ha='right', fontsize=FONT_SIZE - 4)
    plt.yticks(fontsize=FONT_SIZE - 3)
    plt.tight_layout()
    return fig, '05_notch_signaling_heatmap'


def plot_acc_pathway_dotplot(ctx):
    active_pathways = {}
    for pathway_key, pathway_label in ACC_PATHWAYS.items():
        pathway_sig = ctx.sig_means[ctx.sig_means['classification'].str.contains(pathway_key, na=False)]
        n_sig = pathway_sig[ctx.cell_pairs].notna().sum().sum()
        if n_sig > 0:
            active_pathways[pathway_key] = pathway_label

    rows = []
    for pathway_key, pathway_label in active_pathways.items():
        pathway_sig = ctx.sig_means[ctx.sig_means['classification'].str.contains(pathway_key, na=False)]
        pathway_pval = ctx.pvals[ctx.pvals['interacting_pair'].isin(pathway_sig['interacting_pair'])]

        for _, row in pathway_sig.iterrows():
            pair_name = row['interacting_pair']
            pval_row = pathway_pval[pathway_pval['interacting_pair'] == pair_name]
            for cp in ctx.cell_pairs:
                mean_val = row[cp]
                if pd.notna(mean_val):
                    p = pval_row[cp].values[0] if len(pval_row) > 0 else 1.0
                    sender, receiver = cp.split('|')
                    rows.append({
                        'Pathway': pathway_label,
                        'Interaction': pair_name,
                        'Cell Pair': f"{shorten(sender)} → {shorten(receiver)}",
                        'Mean Expression': mean_val,
                        '-log10(p)': -np.log10(p + 1e-10),
                    })

    df_plot = pd.DataFrame(rows)
    if len(df_plot) == 0:
        return None, '06_acc_pathway_dotplot'

    agg = df_plot.groupby(['Pathway', 'Cell Pair']).agg(
        mean_expr=('Mean Expression', 'mean'),
        max_sig=('-log10(p)', 'max'),
        n_interactions=('Interaction', 'count')
    ).reset_index()

    top_list = [agg[agg['Pathway'] == p].nlargest(3, 'n_interactions')
                for p in agg['Pathway'].unique()]
    top_pairs = pd.concat(top_list, ignore_index=True)

    n_rows = len(top_pairs) + len(active_pathways)
    fig_height = max(10, n_rows * 0.55)
    fig, ax = plt.subplots(figsize=(12, fig_height))

    pathways_ordered = list(active_pathways.values())
    y_labels, y_pos, pathway_boundaries = [], [], []
    current_y = 0

    for pathway in pathways_ordered:
        subset = top_pairs[top_pairs['Pathway'] == pathway].sort_values('mean_expr', ascending=False)
        if len(subset) == 0:
            continue
        pathway_boundaries.append((current_y, pathway))
        for _, r in subset.iterrows():
            ax.scatter(r['mean_expr'], current_y, s=r['n_interactions'] * 80,
                       c=[r['max_sig']], cmap='RdYlBu_r', vmin=0, vmax=5,
                       edgecolors='black', linewidths=0.5, zorder=3)
            y_labels.append(f"{r['Cell Pair']}")
            y_pos.append(current_y)
            current_y += 1
        current_y += 1.0

    ax.set_yticks(y_pos)
    ax.set_yticklabels(y_labels, fontsize=FONT_SIZE - 2)
    ax.set_xlabel('Mean Expression (Significant Interactions)', fontsize=FONT_SIZE)
    ax.set_title(ctx.ttl('ACC-Critical Signaling Pathways: Top Cell-Cell Interactions\n'
                         '(Bubble size = # interactions, Color = significance)'),
                 fontsize=FONT_SIZE, fontweight='bold', pad=15)

    for y_start, pathway_name in pathway_boundaries:
        ax.annotate(pathway_name, xy=(ax.get_xlim()[1], y_start),
                    xytext=(10, 0), textcoords='offset points',
                    fontsize=FONT_SIZE - 1, fontweight='bold', va='bottom',
                    ha='left', color='#333333')
    for y_start, _ in pathway_boundaries[1:]:
        ax.axhline(y=y_start - 0.5, color='gray', linewidth=0.5, linestyle='--', alpha=0.4)

    ax.invert_yaxis()
    ax.grid(axis='x', alpha=0.2)

    sm = plt.cm.ScalarMappable(cmap='RdYlBu_r', norm=plt.Normalize(0, 5))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, shrink=0.4, pad=0.14)
    cbar.set_label('-log10(p-value)', fontsize=FONT_SIZE - 1)
    cbar.ax.tick_params(labelsize=FONT_SIZE - 3)

    for s_val, s_label in [(1, '1'), (3, '3'), (5, '5')]:
        ax.scatter([], [], s=s_val * 80, c='gray', edgecolors='black',
                   linewidths=0.5, label=f'{s_label} interactions')
    ax.legend(title='# Interactions', loc='lower right', fontsize=FONT_SIZE - 3,
              title_fontsize=FONT_SIZE - 2)
    plt.tight_layout()
    return fig, '06_acc_pathway_dotplot'


def plot_epithelial_centric_interactions(ctx):
    epi_pairs = [cp for cp in ctx.cell_pairs if 'Epithelial cells' in cp]
    if not epi_pairs:
        return None, '07_epithelial_centric_interactions'

    rows = []
    for _, row in ctx.sig_means.iterrows():
        for cp in epi_pairs:
            val = row[cp]
            if pd.notna(val):
                pval_row = ctx.pvals[ctx.pvals['interacting_pair'] == row['interacting_pair']]
                p = pval_row[cp].values[0] if len(pval_row) > 0 else 1.0
                sender, receiver = cp.split('|')
                partner = receiver if sender == 'Epithelial cells' else sender
                direction = 'Outgoing' if sender == 'Epithelial cells' else 'Incoming'
                rows.append({
                    'Interaction': row['interacting_pair'],
                    'Classification': row['classification'] if pd.notna(row['classification']) else 'Other',
                    'Partner': shorten(partner),
                    'Direction': direction,
                    'Mean': val,
                    'p_value': p,
                    '-log10(p)': -np.log10(p + 1e-10),
                })

    df_epi = pd.DataFrame(rows)
    if len(df_epi) == 0:
        return None, '07_epithelial_centric_interactions'
    top20 = df_epi.nlargest(20, 'Mean').reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(12, 10))
    colors = {'Outgoing': '#5A9BD5', 'Incoming': '#ED7D31'}
    for idx, (_, r) in enumerate(top20.iterrows()):
        ax.scatter(r['Mean'], idx, s=r['-log10(p)'] * 50, c=colors[r['Direction']],
                   alpha=0.8, edgecolors='#333333', linewidths=0.5, zorder=3)

    y_labels = [f"{r['Interaction']}  ({r['Partner']})" for _, r in top20.iterrows()]
    ax.set_yticks(range(len(top20)))
    ax.set_yticklabels(y_labels, fontsize=FONT_SIZE - 3)
    ax.invert_yaxis()
    ax.set_xlabel('Mean Expression', fontsize=FONT_SIZE)
    ax.set_title(ctx.ttl('Top 20 Epithelial Cell Interactions in ACC\n'
                         '(Outgoing = Epithelial as sender, Incoming = Epithelial as receiver)'),
                 fontsize=FONT_SIZE, fontweight='bold', pad=15)
    legend_elements = [
        plt.scatter([], [], s=80, c='#5A9BD5', edgecolors='#333333', linewidths=0.5, label='Outgoing'),
        plt.scatter([], [], s=80, c='#ED7D31', edgecolors='#333333', linewidths=0.5, label='Incoming'),
    ]
    ax.legend(handles=legend_elements, fontsize=FONT_SIZE - 1, loc='lower right')
    ax.grid(True, alpha=0.2, which='both')
    ax.set_axisbelow(True)
    plt.tight_layout()
    return fig, '07_epithelial_centric_interactions'


def plot_pathway_distribution(ctx):
    cat_pairs = {'Tumor-Tumor': [], 'Tumor-Stroma': [], 'Tumor-Immune': [],
                 'Stroma-Stroma': [], 'Stroma-Immune': [], 'Immune-Immune': []}
    for cp in ctx.cell_pairs:
        a, b = cp.split('|')
        cat_a = CELL_CATEGORIES.get(a, 'Other')
        cat_b = CELL_CATEGORIES.get(b, 'Other')
        key, rev_key = f"{cat_a}-{cat_b}", f"{cat_b}-{cat_a}"
        if key in cat_pairs:
            cat_pairs[key].append(cp)
        elif rev_key in cat_pairs:
            cat_pairs[rev_key].append(cp)

    data = {}
    for pathway_key, pathway_label in ACC_PATHWAYS.items():
        pathway_data = ctx.sig_means[ctx.sig_means['classification'].str.contains(pathway_key, na=False)]
        counts, total = {}, 0
        for cat_name, cps in cat_pairs.items():
            c = sum(pathway_data[cp].notna().sum() for cp in cps)
            counts[cat_name] = c
            total += c
        if total > 0:
            data[pathway_label] = counts

    if len(data) == 0:
        return None, '08_pathway_distribution_by_compartment'

    df_bar = pd.DataFrame(data).T
    df_bar['_total'] = df_bar.sum(axis=1)
    df_bar = df_bar.sort_values('_total', ascending=True).drop(columns='_total')

    fig, ax = plt.subplots(figsize=(12, max(6, len(df_bar) * 0.6)))
    cat_colors = {'Tumor-Tumor': '#AEC7E8', 'Tumor-Stroma': '#FFBB78',
                  'Tumor-Immune': '#98DF8A', 'Stroma-Stroma': '#FF9896',
                  'Stroma-Immune': '#C5B0D5', 'Immune-Immune': '#C49C94'}
    bottom = np.zeros(len(df_bar))
    for cat_name in cat_pairs:
        if cat_name in df_bar.columns:
            vals = df_bar[cat_name].values
            ax.barh(range(len(df_bar)), vals, left=bottom,
                    color=cat_colors.get(cat_name, '#CCCCCC'), label=cat_name,
                    edgecolor='white', linewidth=0.8)
            bottom += vals

    ax.set_yticks(range(len(df_bar)))
    ax.set_yticklabels(df_bar.index, fontsize=FONT_SIZE - 1)
    ax.set_xlabel('Number of Significant Interactions', fontsize=FONT_SIZE)
    ax.set_title(ctx.ttl('Distribution of ACC-Relevant Signaling Pathways\n'
                         'Across TME Compartments'),
                 fontsize=FONT_SIZE, fontweight='bold', pad=15)
    ax.legend(fontsize=FONT_SIZE - 3, loc='lower right', title='Compartment Pair',
              title_fontsize=FONT_SIZE - 2, frameon=True, fancybox=True, framealpha=0.9)
    ax.grid(axis='x', alpha=0.2)
    ax.set_axisbelow(True)
    ax.tick_params(axis='x', labelsize=FONT_SIZE - 2)
    plt.tight_layout()
    return fig, '08_pathway_distribution_by_compartment'


def plot_top_lr_pairs_dotplot(ctx):
    sig_copy = ctx.sig_means.copy()
    sig_copy['n_sig_pairs'] = sig_copy[ctx.cell_pairs].notna().sum(axis=1)
    sig_copy['mean_sig'] = sig_copy[ctx.cell_pairs].mean(axis=1, skipna=True)
    top10 = sig_copy[sig_copy['n_sig_pairs'] > 0].nlargest(10, 'n_sig_pairs')
    if len(top10) == 0:
        return None, '09_top10_LR_pairs_dotplot'

    active_cp = [cp for cp in ctx.cell_pairs if top10[cp].notna().any()]
    cp_activity = {cp: top10[cp].notna().sum() for cp in active_cp}
    active_cp = sorted(cp_activity, key=cp_activity.get, reverse=True)[:20]

    plot_means = top10.set_index('interacting_pair')[active_cp].astype(float)
    pval_subset = ctx.pvals[ctx.pvals['interacting_pair'].isin(top10['interacting_pair'])]
    plot_pvals = pval_subset.set_index('interacting_pair')[active_cp].astype(float)

    def shorten_pair_label(cp):
        a, b = cp.split('|')
        return f"{shorten(a)} → {shorten(b)}"
    col_labels = [shorten_pair_label(c) for c in active_cp]

    fig, ax = plt.subplots(figsize=(16, 8))
    for i, interaction in enumerate(plot_means.index):
        for j, cp in enumerate(active_cp):
            mean_val = plot_means.loc[interaction, cp]
            if pd.isna(mean_val):
                continue
            pval = plot_pvals.loc[interaction, cp] if interaction in plot_pvals.index else 1.0
            if pd.isna(pval):
                pval = 1.0
            ax.scatter(j, i, s=mean_val * 100, c=-np.log10(pval + 1e-10),
                       cmap='RdYlBu_r', vmin=0, vmax=5,
                       edgecolors='black', linewidths=0.4, zorder=3)

    ax.set_xticks(range(len(col_labels)))
    ax.set_xticklabels(col_labels, rotation=90, fontsize=FONT_SIZE - 4, ha='center')
    ax.set_yticks(range(len(plot_means.index)))
    ax.set_yticklabels(plot_means.index, fontsize=FONT_SIZE - 2)
    ax.set_xlim(-0.5, len(col_labels) - 0.5)
    ax.set_ylim(-0.5, len(plot_means.index) - 0.5)
    ax.invert_yaxis()
    ax.grid(True, alpha=0.15)
    ax.set_axisbelow(True)
    ax.set_title(ctx.ttl('Top 10 Ligand-Receptor Interactions in ACC\n'
                         '(Bubble size = expression, Color = significance)'),
                 fontsize=FONT_SIZE, fontweight='bold', pad=15)
    ax.set_xlabel('Cell-Cell Pair (Sender → Receiver)', fontsize=FONT_SIZE)
    ax.set_ylabel('Ligand-Receptor Pair', fontsize=FONT_SIZE)

    sm = plt.cm.ScalarMappable(cmap='RdYlBu_r', norm=plt.Normalize(0, 5))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, shrink=0.6, pad=0.02)
    cbar.set_label('-log10(p-value)', fontsize=FONT_SIZE - 1)
    cbar.ax.tick_params(labelsize=FONT_SIZE - 3)

    for s_val in [0.5, 1.0, 1.5, 2.0]:
        ax.scatter([], [], s=s_val * 100, c='gray', edgecolors='black',
                   linewidths=0.4, label=f'{s_val:.1f}')
    ax.legend(title='Mean Expr.', loc='upper right', fontsize=FONT_SIZE - 4,
              title_fontsize=FONT_SIZE - 3, bbox_to_anchor=(1.18, 1.0))
    plt.tight_layout()
    return fig, '09_top10_LR_pairs_dotplot'

In [ ]:
# =============================================================================
# B1 driver - run the descriptive plot suite three times.
# Replaces the legacy `--base-dir` argument and the missing Good-group run.
# =============================================================================
CPDB_PLOT_RUNS = [
    ("all_samples", ALL_SAMPLES_DIR, FIG_POOLED_ALL,
     f"{EXPECTED_N_PATIENTS} patients",
     "All samples — descriptive map (not an outcome comparison)"),
    ("pooled_good", POOLED_GOOD_DIR, FIG_POOLED_GOOD,
     f"pooled Good-outcome cells, {EXPECTED_N_GOOD} patients",
     f"{POOLED_SUBTITLE} — Good clinical-outcome group"),
    ("pooled_poor", POOLED_POOR_DIR, FIG_POOLED_POOR,
     f"pooled Poor-outcome cells, {EXPECTED_N_POOR} patients",
     f"{POOLED_SUBTITLE} — Poor clinical-outcome group"),
]

cpdb_plot_report = []
def assert_pathway_keys_unambiguous(ctx):
    '''Each ACC_PATHWAYS key must select at most one classification.

    The legacy filter is `classification.str.contains(key, na=False)` -- CASE
    SENSITIVE. On the current database all eight keys resolve 1:1, so the
    pathway figures do not merge distinct pathways. (Note 'Ephrin' does NOT
    match 'Signaling by Epinephrine': the substring there is 'nephrine' and the
    match is case-sensitive.) A key matching zero classifications is fine -- the
    pathway simply has no significant means in that cohort. A key matching two
    or more is a real collision introduced by a database change, and is fatal.
    '''
    cls = pd.Series(sorted(set(ctx.sig_means['classification']
                               .dropna().astype(str).str.strip())))
    matches = {k: cls[cls.str.contains(k, na=False)].tolist() for k in ACC_PATHWAYS}
    ambiguous = {k: v for k, v in matches.items() if len(v) > 1}
    assert not ambiguous, (
        "an ACC_PATHWAYS key selects more than one classification, so the pathway "
        f"figures would merge distinct pathways: {ambiguous}")
    return {k: (v[0] if v else None) for k, v in matches.items()}


for run_name, base_dir, out_dir, cohort_label, subtitle in CPDB_PLOT_RUNS:
    assert base_dir.is_dir(), f"missing CellPhoneDB run directory: {base_dir}"
    ctx = make_ctx(base_dir, out_dir, cohort_label, subtitle)
    pathway_key_matches = assert_pathway_keys_unambiguous(ctx)

    produced, skipped = [], []
    single_plots = [plot_significant_interaction_heatmap, plot_mean_interaction_heatmap,
                    plot_network_diagram, plot_notch_pathway_heatmap,
                    plot_acc_pathway_dotplot, plot_epithelial_centric_interactions,
                    plot_pathway_distribution, plot_top_lr_pairs_dotplot]
    for fn in single_plots:
        fig, name = fn(ctx)
        if fig is None:
            skipped.append(name)
            continue
        save_fig(fig, out_dir, name, base_dir / 'significant_means.csv',
                 f"exploratory_pooled/{run_name}")
        produced.append(name)

    for fig, name in plot_chord_diagrams(ctx):
        if fig is None:
            skipped.append(name)
            continue
        save_fig(fig, out_dir, name, base_dir / 'significant_means.csv',
                 f"exploratory_pooled/{run_name}")
        produced.append(name)

    cpdb_plot_report.append({"run": run_name, "base_dir": str(base_dir),
                             "out_dir": str(out_dir),
                             "n_cell_types": len(ctx.cell_types),
                             "n_interactions": int(len(ctx.sig_means)),
                             "n_figures": len(produced),
                             "skipped": ";".join(skipped),
                             "pathway_keys_unambiguous": True,
                             "pathway_key_matches": pathway_key_matches})
    assert len(produced) >= 6, f"{run_name}: only {len(produced)} descriptive figures produced"

cpdb_plot_report_df = pd.DataFrame(cpdb_plot_report)
_log(event="cpdb_descriptive_plots", runs=cpdb_plot_report)
cpdb_plot_report_df

## B2 — Pooled differential figures (`plot_figures.py`)

Two required fixes, both applied below.

**1. Paths.** The legacy script sets `INPUT_DIR = OUT_DIR = SCRIPT_DIR`, which only works
because the script physically sits inside its data folder. Inputs and outputs are now
separate explicit config paths: it reads from `exploratory_pooled/differential_enrichment_results/`
and writes to `figures/exploratory_pooled/differential/`.

**2. Deletion safety.** Legacy lines 462–475 unlink 14 hardcoded `STALE` filenames from
`OUT_DIR` on every run, and lines 505–508 shell out to `grep` for the SVG self-check.
With `OUT_DIR` derived from `__file__`, an unlink list like that is one misplaced copy of
the script away from deleting files in an arbitrary directory.

* Every deletion now goes through `safe_unlink(path, allowed_root)` with
  `allowed_root = OUTPUT_ROOT`. A path outside the new output root cannot be removed.
* Only files this notebook itself has just written are ever candidates; the deletion
  target directory is asserted to be inside `figures/exploratory_pooled/differential/`
  before the loop runs.
* The `grep` subprocess is replaced by an in-Python read of the SVG text.

The lollipop layout, `ABBREV`, the good/poor/both colours and `STATUS_ORDER` are unchanged.

In [ ]:
# =============================================================================
# PORTED from legacy_sources/plot_figures.py
#   - INPUT_DIR/OUT_DIR: explicit config paths (was SCRIPT_DIR)
#   - STALE unlinks: routed through safe_unlink(path, allowed_root)
#   - grep self-check: replaced with an in-Python check
#   - ABBREV / colours / STATUS_ORDER: verbatim
# =============================================================================

ABBREV = {
    "ADC - Tumor":                     "ADC - Tumor",
    "ADMEC - Tumor":                   "ADMEC - Tumor",
    "Epithelial cells - Tumor":        "Epithelial - Tumor",
    "Fibroblast cells":                "Fibroblast",
    "Fibroblast cells - CAF":          "Fibroblast - CAF",
    "Fibroblast cells - Inflammatory": "Fibroblast - Inflam",
    "Macrophages - AP/TAM":            "Macrophage - TAM",
    "Macrophages - M2":                "Macrophage M2",
    "B cells":                         "B cells",
    "Memory B cells":                  "Memory B",
    "CD4+ T cells":                    "CD4+ T",
    "CD8+ T cells":                    "CD8+ T",
    "Plasma cells":                    "Plasma",
    "Endothelial cells":               "Endothelial",
    "Epithelial cells":                "Epithelial",
    "Mast cells":                      "Mast",
    "Mural cells":                     "Mural",
    "Muscle cells":                    "Muscle",
}


def abbrev_pair(pair_str):
    parts = [p.strip() for p in pair_str.split("|")]
    return " | ".join(ABBREV.get(p, p) for p in parts)


def load_wilcoxon(input_dir, fname):
    df = pd.read_csv(Path(input_dir) / fname)
    df["neg_log10_fdr"] = -np.log10(df["fdr"].clip(lower=1e-300))
    df["significant"]   = df["fdr"] < FDR_ALPHA
    return df


def plot_lollipop_directional(scores_df, direction, color, n=10):
    '''Top-N interactions for one enrichment direction, ranked by FDR.
    Layout verbatim from the legacy script.'''
    X_CLIP = 5
    X_PAD  = 1.0

    sig = scores_df[scores_df["significant"] & (scores_df["direction"] == direction)]
    if len(sig) == 0:
        return None
    top = (sig.nsmallest(n, "fdr")
              .sort_values("log2fc_good_vs_poor", ascending=True)
              .reset_index(drop=True))

    top["l2fc_plot"] = top["log2fc_good_vs_poor"].clip(-X_CLIP, X_CLIP)
    top["clipped"]   = top["log2fc_good_vs_poor"].abs() > X_CLIP

    nlf = top["neg_log10_fdr"]
    rng = nlf.max() - nlf.min()
    top["dot_size"] = 60 + (nlf - nlf.min()) / (rng + 1e-9) * 290

    fig, ax = plt.subplots(figsize=(8, 5.6))
    y = np.arange(len(top))

    for i, row in top.iterrows():
        ax.plot([0, row["l2fc_plot"]], [i, i], color=color, lw=2.0, alpha=0.75,
                zorder=1, solid_capstyle="butt")

    nc = top[~top["clipped"]]
    if len(nc):
        ax.scatter(nc["l2fc_plot"], nc.index, s=nc["dot_size"], c=color,
                   edgecolors="white", linewidths=0.8, zorder=3)

    cl = top[top["clipped"] & (top["log2fc_good_vs_poor"] < 0)]
    if len(cl):
        ax.scatter(cl["l2fc_plot"], cl.index, s=cl["dot_size"] * 1.3, c=color,
                   marker="<", edgecolors="white", linewidths=0.8, zorder=3)
        for _, row in cl.iterrows():
            ax.text(-X_CLIP - 0.12, row.name,
                    "  {:.1f}".format(row["log2fc_good_vs_poor"]),
                    va="center", ha="left", fontsize=9, color="#444444", style="italic")

    cr = top[top["clipped"] & (top["log2fc_good_vs_poor"] > 0)]
    if len(cr):
        ax.scatter(cr["l2fc_plot"], cr.index, s=cr["dot_size"] * 1.3, c=color,
                   marker=">", edgecolors="white", linewidths=0.8, zorder=3)
        for _, row in cr.iterrows():
            ax.text(X_CLIP + 0.12, row.name,
                    "{:.1f}  ".format(row["log2fc_good_vs_poor"]),
                    va="center", ha="right", fontsize=9, color="#444444", style="italic")

    ax.axvline(0, color="#444444", lw=0.9, zorder=2)
    ax.set_xlim(-X_CLIP - X_PAD, X_CLIP + X_PAD)
    ax.set_ylim(-0.8, len(top) - 0.2)
    ax.set_yticks(y)
    ax.set_yticklabels(top["interacting_pair"], fontsize=11)
    ax.set_xlabel("log₂FC  (Good / Poor)", fontsize=13)

    label = "Good-Enriched" if direction == "higher_in_good" else "Poor-Enriched"
    ax.set_title("Top {}  {} Ligand–Receptor Interactions\n"
                 "Wilcoxon on interaction scores (ranked by FDR)\n"
                 "{}".format(n, label, POOLED_SUBTITLE),
                 fontsize=14, fontweight="bold", pad=10)

    fdr_levels = [1e-2, 1e-5, 1e-10]
    display_sizes = [7, 11, 16]
    size_handles = [
        Line2D([0], [0], marker="o", linestyle="None", markerfacecolor=color,
               markeredgecolor="white", markeredgewidth=0.8, markersize=ds,
               label="FDR = {:.0e}".format(fdr))
        for ds, fdr in zip(display_sizes, fdr_levels)
    ]
    ax.legend(handles=size_handles, title="−log₁₀(FDR) → dot size",
              title_fontsize=10,
              loc="lower right" if direction == "higher_in_poor" else "lower left",
              framealpha=0.92, edgecolor="#cccccc", fontsize=10, handletextpad=1.0)
    sns.despine(ax=ax, left=True)
    fig.tight_layout()
    return fig


def draw_stacked_bar_donut(detail, groupby_col, label_fn, title_bar,
                           n_top=20, bar_label_fs=9, ytick_fs=10):
    '''Two-panel figure: stacked bar (top-N groups) + donut (global proportions).
    Verbatim from the legacy script apart from the subtitle.'''
    top_groups = detail.groupby(groupby_col).size().nlargest(n_top).index.tolist()
    sub = detail[detail[groupby_col].isin(top_groups)].copy()
    pivot = (sub.groupby([groupby_col, "significance_status"]).size()
                .unstack(fill_value=0)
                .reindex(columns=STATUS_ORDER, fill_value=0))
    pivot["_tot"] = pivot.sum(axis=1)
    pivot = pivot.sort_values("_tot").drop(columns="_tot")
    pivot.index = [label_fn(x) for x in pivot.index]

    gc       = detail["significance_status"].value_counts()
    d_vals   = [gc.get(s, 0) for s in STATUS_ORDER]
    d_colors = [STATUS_COLORS[s] for s in STATUS_ORDER]
    total_n  = sum(d_vals)

    fig     = plt.figure(figsize=(16, 8), facecolor="white")
    ax_bar  = fig.add_axes([0.04, 0.08, 0.58, 0.80])
    ax_dnut = fig.add_axes([0.68, 0.12, 0.30, 0.72])

    y_pos = np.arange(len(pivot))
    lefts = np.zeros(len(pivot))
    for status in STATUS_ORDER:
        vals = pivot[status].values.astype(float)
        ax_bar.barh(y_pos, vals, left=lefts, color=STATUS_COLORS[status],
                    label=STATUS_LABELS[status], edgecolor="white", linewidth=0.6)
        for j, (v, l) in enumerate(zip(vals, lefts)):
            if v >= 5:
                ax_bar.text(l + v / 2, j, str(int(v)), ha="center", va="center",
                            fontsize=bar_label_fs, color="white", fontweight="bold")
        lefts += vals

    ax_bar.set_yticks(y_pos)
    ax_bar.set_yticklabels(pivot.index, fontsize=ytick_fs)
    ax_bar.set_xlabel("Number of Interactions", fontsize=13)
    ax_bar.set_title(f"{title_bar}\n{POOLED_SUBTITLE}",
                     fontsize=14, fontweight="bold", pad=10)
    ax_bar.legend(loc="lower right", framealpha=0.9, edgecolor="#cccccc", fontsize=11)
    sns.despine(ax=ax_bar, left=True)

    wedges, _ = ax_dnut.pie(d_vals, colors=d_colors, startangle=90,
                            wedgeprops=dict(width=0.52, edgecolor="white", linewidth=1.8),
                            counterclock=False)
    ax_dnut.text(0,  0.10, "Total", ha="center", va="center",
                 fontsize=12, color="#666666")
    ax_dnut.text(0, -0.14, "n={:,}".format(total_n), ha="center", va="center",
                 fontsize=15, fontweight="bold", color="#222222")

    for wedge, status, val in zip(wedges, STATUS_ORDER, d_vals):
        angle  = (wedge.theta2 + wedge.theta1) / 2
        rad    = np.radians(angle)
        cx, cy = np.cos(rad) * 0.78, np.sin(rad) * 0.78
        tx, ty = np.cos(rad) * 1.28, np.sin(rad) * 1.28
        ha     = "left" if tx > 0 else "right"
        pct    = val / total_n * 100 if total_n else 0.0
        ax_dnut.annotate("{}\n{:,}  ({:.1f}%)".format(STATUS_LABELS[status], val, pct),
                         xy=(cx, cy), xytext=(tx, ty),
                         arrowprops=dict(arrowstyle="-", color="#999999", lw=0.8),
                         ha=ha, va="center", fontsize=11)

    ax_dnut.set_title("Overall Significance\nDistribution",
                      fontsize=13, fontweight="bold", pad=14)
    return fig

In [ ]:
# =============================================================================
# B2 driver + contained cleanup + in-Python verification
# =============================================================================
scores_df = load_wilcoxon(DE_RESULTS_DIR, "wilcoxon_per_interaction_scores.csv")
detail_df = pd.read_csv(DE_RESULTS_DIR / "detailed_interaction_summary.csv")
assert len(scores_df) > 0 and len(detail_df) > 0

diff_written = []
for direction, color, figname in [
    ("higher_in_poor", C_POOR, "fig2a_lollipop_poor_enriched"),
    ("higher_in_good", C_GOOD, "fig2b_lollipop_good_enriched"),
]:
    fig = plot_lollipop_directional(scores_df, direction, color)
    if fig is None:
        _log(event="lollipop_skipped", direction=direction,
             reason="no interaction reached FDR < 0.05 in this direction")
        continue
    diff_written += save_fig(fig, FIG_POOLED_DIFF, figname,
                             DE_RESULTS_DIR / "wilcoxon_per_interaction_scores.csv",
                             "exploratory_pooled/differential")

fig = draw_stacked_bar_donut(
    detail_df, "cell_type_pair", abbrev_pair,
    "Top 20 Cell-Type Communication Axes\nSignificance by group  (Good vs. Poor)",
    ytick_fs=11)
diff_written += save_fig(fig, FIG_POOLED_DIFF, "fig6_overview_cellpair",
                         DE_RESULTS_DIR / "detailed_interaction_summary.csv",
                         "exploratory_pooled/differential")

MAX_CLS_LEN = 40


def shorten_cls(name):
    name = str(name)
    return name if len(name) <= MAX_CLS_LEN else name[:MAX_CLS_LEN - 1] + "…"


fig = draw_stacked_bar_donut(
    detail_df, "classification", shorten_cls,
    "Top 20 Interaction Classifications\nSignificance by group  (Good vs. Poor)",
    ytick_fs=10)
diff_written += save_fig(fig, FIG_POOLED_DIFF, "fig7_overview_classification",
                         DE_RESULTS_DIR / "detailed_interaction_summary.csv",
                         "exploratory_pooled/differential")

# --- CONTAINED CLEANUP ------------------------------------------------------
# Legacy lines 272-277 and 462-475 removed 14+2 hardcoded filenames from a
# directory derived from __file__. Every removal here is scoped to
# FIG_POOLED_DIFF and passes through safe_unlink(path, allowed_root).
STALE = [
    "fig1_volcano_interaction_scores.png",   "fig1_volcano_interaction_scores.svg",
    "fig2_lollipop_top40_scores.png",        "fig2_lollipop_top40_scores.svg",
    "fig2_lollipop_top20_scores.png",        "fig2_lollipop_top20_scores.svg",
    "fig3_heatmap_cellpair_scores.png",      "fig3_heatmap_cellpair_scores.svg",
    "fig4_pathway_bubble.png",               "fig4_pathway_bubble.svg",
    "fig5_volcano_interaction_means.png",    "fig5_volcano_interaction_means.svg",
    "fig6_overview_significance_status.png", "fig6_overview_significance_status.svg",
]

_target = FIG_POOLED_DIFF.resolve()
assert str(_target).startswith(str(FIG_ROOT.resolve())), \
    "stale-file cleanup target is outside the figures tree; refusing to delete"
assert str(_target).startswith(str(DELETION_ALLOWED_ROOT.resolve())), \
    "stale-file cleanup target is outside the new output root; refusing to delete"

removed = []
for f in STALE:
    p = _target / f
    if p.exists():
        safe_unlink(p, DELETION_ALLOWED_ROOT)
        removed.append(f)

_log(event="stale_cleanup", target_dir=str(_target),
     allowed_root=str(DELETION_ALLOWED_ROOT), n_removed=len(removed), removed=removed,
     mechanism="utils.safe_unlink(path, allowed_root)")

# --- IN-PYTHON VERIFICATION (replaces the legacy `grep -c "<text"` subprocess) ---
EXPECTED = [f"{n}.{e}" for n in
            ["fig2a_lollipop_poor_enriched", "fig2b_lollipop_good_enriched",
             "fig6_overview_cellpair", "fig7_overview_classification"]
            for e in ("png", "svg")]

verification = []
produced_names = {p.name for p in diff_written}
for fname in EXPECTED:
    p = FIG_POOLED_DIFF / fname
    exists = p.is_file()
    size_kb = p.stat().st_size / 1024 if exists else 0.0
    if not exists and fname not in produced_names:
        verification.append({"file": fname, "exists": False, "size_kb": 0.0,
                             "svg_has_text": None, "ok": False,
                             "note": "not produced (no hits in that direction)"})
        continue
    if p.suffix == ".svg":
        has_text = "<text" in p.read_text(encoding="utf-8", errors="ignore")
        ok = exists and has_text
    else:
        has_text = None
        ok = exists and size_kb >= 50
    verification.append({"file": fname, "exists": exists, "size_kb": round(size_kb, 1),
                         "svg_has_text": has_text, "ok": ok, "note": ""})

verification_df = pd.DataFrame(verification)
verification_df.to_csv(FIG_POOLED_DIFF / "figure_verification.csv", index=False)
assert verification_df.loc[verification_df["note"] == "", "ok"].all(), \
    f"figure verification failed:\n{verification_df}"
verification_df

## B3 — Consistency figures (`plot_consistency.py`)

`BASE = os.path.dirname(os.path.abspath(__file__))` is replaced by explicit config
paths: input `exploratory_pooled/consistency_results/consistency_summary.csv`, output
`figures/exploratory_pooled/consistency/`. The two figures and their palettes are
unchanged.

These panels describe how the pooled hits behave across individual patients. They are a
consistency/reproducibility description, not a statistical test, and they are not a
patient-level outcome analysis.

In [ ]:
# =============================================================================
# PORTED from legacy_sources/plot_consistency.py
#   - BASE (derived from __file__) replaced by explicit config paths
# =============================================================================
COL_CONSIST   = '#7B1E3A'
COL_INCONSIST = '#2A7B8C'
COL_CONS_SC   = '#4A8C5C'
COL_INC_SC    = '#C03535'


def build_consistency_figures(summary_csv, out_dir):
    summary = pd.read_csv(summary_csv)
    summary['neg_log10_fdr'] = -np.log10(summary['pooled_fdr'].clip(lower=1e-30))
    written = []

    # -- Figure 1: donut + stacked bar ---------------------------------------
    n_consist   = int(summary['consistent_with_pooled'].sum())
    n_inconsist = int((~summary['consistent_with_pooled'].astype(bool)).sum())
    n_total     = len(summary)

    fig1, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(13, 5.9),
                                      gridspec_kw={'width_ratios': [1, 1.1]})
    fig1.subplots_adjust(left=0.04, right=0.98, top=0.82, bottom=0.14, wspace=0.28)
    fig1.suptitle(POOLED_SUBTITLE, fontsize=12, style='italic',
                  color='#555555', y=0.98)

    for lbl, title, ax in zip(['A', 'B'],
                              ['Overall consistency', 'Consistency by pooled direction'],
                              [ax_a, ax_b]):
        ax.text(0.0, 1.08, lbl, transform=ax.transAxes,
                fontsize=15, fontweight='bold', va='top', ha='left')
        ax.text(0.07, 1.08, title, transform=ax.transAxes,
                fontsize=13, fontweight='bold', va='top', ha='left')

    wedges, _ = ax_a.pie([n_consist, n_inconsist],
                         colors=[COL_CONSIST, COL_INCONSIST],
                         wedgeprops=dict(width=0.44, edgecolor='white', linewidth=2.5),
                         startangle=90, counterclock=False)
    for wedge, n in zip(wedges, [n_consist, n_inconsist]):
        ang = np.radians((wedge.theta1 + wedge.theta2) / 2)
        ax_a.text(0.72 * np.cos(ang), 0.72 * np.sin(ang),
                  f'{n / n_total * 100:.1f}%', ha='center', va='center',
                  fontsize=13, fontweight='bold', color='white')
    ax_a.text(0,  0.10, str(n_total), ha='center', va='center',
              fontsize=24, fontweight='bold', color='#2a2a2a')
    ax_a.text(0, -0.14, 'Interactions', ha='center', va='center',
              fontsize=11, color='#666666')
    ax_a.set_aspect('equal')
    ax_a.legend(handles=[mpatches.Patch(color=COL_CONSIST,   label=f'Consistent  (n={n_consist})'),
                         mpatches.Patch(color=COL_INCONSIST, label=f'Inconsistent (n={n_inconsist})')],
                frameon=False, loc='lower center', bbox_to_anchor=(0.5, -0.13),
                fontsize=11, handlelength=1.2, handleheight=1.0)

    dir_order  = ['higher_in_good', 'higher_in_poor']
    dir_labels = ['Higher in\nGood Outcome', 'Higher in\nPoor Outcome']
    dir_colors = ['#4A8C5C', '#C03535']

    ct = (summary.groupby(['pooled_direction', 'consistent_with_pooled']).size()
                 .unstack(fill_value=0).reindex(dir_order, fill_value=0))
    for c in [True, False]:
        if c not in ct.columns:
            ct[c] = 0
    cons_v   = ct[True].values.astype(float)
    incons_v = ct[False].values.astype(float)
    x = np.arange(len(dir_order))

    ax_b.bar(x, cons_v, width=0.46, color=COL_CONSIST, label='Consistent',
             edgecolor='white', linewidth=0.5)
    ax_b.bar(x, incons_v, width=0.46, color=COL_INCONSIST, label='Inconsistent',
             edgecolor='white', linewidth=0.5, bottom=cons_v)
    for i, (c, ic) in enumerate(zip(cons_v, incons_v)):
        if c >= 2:
            ax_b.text(i, c / 2, int(c), ha='center', va='center',
                      fontsize=13, fontweight='bold', color='white')
        if ic >= 2:
            ax_b.text(i, c + ic / 2, int(ic), ha='center', va='center',
                      fontsize=13, fontweight='bold', color='white')
    ax_b.set_xticks(x)
    ax_b.set_xticklabels(dir_labels, fontsize=12)
    for tick, c in zip(ax_b.get_xticklabels(), dir_colors):
        tick.set_color(c)
        tick.set_fontweight('bold')
    ax_b.set_ylabel('Number of interactions', fontsize=12)
    ax_b.set_ylim(0, max((cons_v + incons_v).max() * 1.16, 1))
    ax_b.spines[['top', 'right']].set_visible(False)
    ax_b.legend(frameon=False, fontsize=11, loc='upper right', handlelength=1.2)

    written += save_fig(fig1, out_dir, 'fig1_consistency_summary',
                        summary_csv, "exploratory_pooled/consistency")

    # -- Figure 2: score & breadth scatter ------------------------------------
    dot_sizes = 20 + summary['neg_log10_fdr'] * 14
    DIRECTION_MARKER = {'higher_in_good': '^', 'higher_in_poor': 'v'}

    def scatter_panel(ax, xcol, ycol, xlabel, ylabel, title, show_legend=False):
        xdata, ydata = summary[xcol], summary[ycol]
        pad = max(xdata.max(), ydata.max()) * 0.06
        lim = max(xdata.max(), ydata.max()) + pad
        for direction, consistent in [('higher_in_good', True), ('higher_in_good', False),
                                      ('higher_in_poor', True), ('higher_in_poor', False)]:
            sub = summary[(summary.pooled_direction == direction) &
                          (summary.consistent_with_pooled == consistent)]
            if sub.empty:
                continue
            ax.scatter(sub[xcol], sub[ycol],
                       c=COL_CONS_SC if consistent else COL_INC_SC,
                       marker=DIRECTION_MARKER[direction],
                       s=dot_sizes.loc[sub.index], alpha=0.82, lw=0.3,
                       edgecolors='white', zorder=3)
        ax.plot([0, lim], [0, lim], color='#999999', lw=1.0, ls='--', zorder=1)
        ax.set_xlim(-pad * 0.5, lim)
        ax.set_ylim(-pad * 0.5, lim)
        ax.set_xlabel(xlabel, fontsize=12)
        ax.set_ylabel(ylabel, fontsize=12)
        ax.set_title(title, fontsize=13, fontweight='bold', pad=8)
        ax.spines[['top', 'right']].set_visible(False)
        if show_legend:
            ax.legend(handles=[
                mpatches.Patch(color=COL_CONS_SC, label='Consistent with pooled'),
                mpatches.Patch(color=COL_INC_SC,  label='Inconsistent with pooled'),
                Line2D([0], [0], marker='^', color='#555', linestyle='none',
                       markersize=9, label='Direction: higher in good'),
                Line2D([0], [0], marker='v', color='#555', linestyle='none',
                       markersize=9, label='Direction: higher in poor'),
                Line2D([0], [0], marker='o', color='#aaa', linestyle='none',
                       markersize=7, label='Size ∝ −log₁₀(FDR)'),
            ], frameon=False, fontsize=11, loc='upper left', handlelength=1.2,
                labelspacing=0.55, borderaxespad=0.3)

    fig2, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(14, 6.4))
    fig2.subplots_adjust(left=0.08, right=0.97, top=0.86, bottom=0.14, wspace=0.36)
    fig2.suptitle(POOLED_SUBTITLE, fontsize=12, style='italic', color='#555555', y=0.99)

    scatter_panel(ax_l, 'mean_score_all_good', 'mean_score_all_poor',
                  'Mean interaction score — Good outcome patients',
                  'Mean interaction score — Poor outcome patients',
                  'Interaction score magnitude: Good vs Poor outcome', show_legend=True)
    scatter_panel(ax_r, 'mean_nsig_pairs_good', 'mean_nsig_pairs_poor',
                  'Mean no. of significant cell-type pairs — Good outcome patients',
                  'Mean no. of significant cell-type pairs — Poor outcome patients',
                  'Significant cell-type pair contexts: Good vs Poor outcome')

    written += save_fig(fig2, out_dir, 'fig2_score_scatter',
                        summary_csv, "exploratory_pooled/consistency")
    return written, n_total, n_consist


consistency_csv = CONSISTENCY_DIR / "consistency_summary.csv"
assert consistency_csv.is_file(), f"{consistency_csv} not found - run 05_ first"
consistency_written, n_consistency_total, n_consistency_consistent = \
    build_consistency_figures(consistency_csv, FIG_POOLED_CONSIST)

_log(event="consistency_figures", n_interactions=n_consistency_total,
     n_consistent=n_consistency_consistent,
     outputs=[str(p) for p in consistency_written])
print(f"consistency figures: {len(consistency_written)} files")

## B4 — Seed-stability figure

The pooled Good/Poor comparison downsamples each retained cell type to
`min(n_good, n_poor)` cells. Seed 0 is the main exploratory run; seeds 1–4 repeat the
downsampling. This figure shows how much of the pooled picture is a property of one
particular draw of cells rather than of the outcome groups.

`05_pooled_exploratory_analysis.ipynb` writes one paired-Wilcoxon table per seed at
`exploratory_pooled/seed_{s}/wilcoxon_per_interaction_scores.csv`. Those tables are the
source for both panels, so the figure regenerates from saved tables and is directly
comparable to the stability statistics computed in `06_sensitivity_analysis.ipynb`
(which reads the same files).

Left panel: how many of the five seeds selected each interaction at FDR < 0.05.
Right panel: pairwise Jaccard overlap of the selected sets.

Seed stability measures sensitivity to cell downsampling. It says nothing about
patient-level validity, and an interaction selected in all five seeds is still an
exploratory pooled-cell signal.

In [ ]:
# =============================================================================
# B4 - seed-stability figure
# =============================================================================
# 05_ writes one paired-Wilcoxon table per downsampling seed at
# exploratory_pooled/seed_{s}/wilcoxon_per_interaction_scores.csv. Those tables
# are the source here, so this figure is reproducible from saved tables and is
# directly comparable to the stability statistics in 06_.
# =============================================================================
seed_tables = {}
for s in POOLED_SEEDS:
    p = EXPLORATORY_DIR / f"seed_{s}" / "wilcoxon_per_interaction_scores.csv"
    if p.is_file():
        seed_tables[s] = p
assert len(seed_tables) >= 2, (
    "seed stability needs at least two per-seed pooled tables under "
    f"{EXPLORATORY_DIR}/seed_*/. Run 05_pooled_exploratory_analysis.ipynb first.")

seed_frames = {s: pd.read_csv(p) for s, p in seed_tables.items()}
selected = {s: set(d.loc[d["fdr"] < FDR_ALPHA, "id_cp_interaction"])
            for s, d in seed_frames.items()}

freq = (pd.concat([d[["id_cp_interaction"]].assign(
            seed=s, selected=d["fdr"] < FDR_ALPHA)
            for s, d in seed_frames.items()], ignore_index=True)
        .groupby("id_cp_interaction")
        .agg(n_seeds_present=("seed", "nunique"),
             n_seeds_selected=("selected", "sum"))
        .reset_index())
freq["selection_frequency"] = freq["n_seeds_selected"] / len(seed_frames)
freq["analysis_scope"] = "exploratory_pooled"
freq_path = FIG_POOLED_SEEDS / "seed_selection_frequency.csv"
freq.to_csv(freq_path, index=False)

seeds_sorted = sorted(seed_tables)
jac = pd.DataFrame(
    [[(len(selected[a] & selected[b]) / len(selected[a] | selected[b]))
      if (selected[a] | selected[b]) else np.nan
      for b in seeds_sorted] for a in seeds_sorted],
    index=seeds_sorted, columns=seeds_sorted)
jac_path = FIG_POOLED_SEEDS / "seed_stability_jaccard.csv"
jac.to_csv(jac_path, index_label="seed")

fig, axes = plt.subplots(1, 2, figsize=(14, 5.8),
                         gridspec_kw={"width_ratios": [1.25, 1]})
sel_only = freq[freq["n_seeds_selected"] > 0]
axes[0].hist(sel_only["n_seeds_selected"],
             bins=np.arange(0.5, len(seed_frames) + 1.5, 1),
             color="#6A8CAF", edgecolor="white")
axes[0].set_xticks(range(1, len(seed_frames) + 1))
axes[0].set_xlabel(f"Number of downsampling seeds selecting the interaction "
                   f"(of {len(seed_frames)})")
axes[0].set_ylabel("Number of interactions")
axes[0].set_title(f"Selection stability at FDR < {FDR_ALPHA}\\n"
                  f"{len(sel_only)} interactions selected in at least one seed",
                  fontweight="bold")
sns.despine(ax=axes[0])

sns.heatmap(jac, annot=True, fmt=".3f", cmap="viridis", vmin=0, vmax=1,
            ax=axes[1], square=True, cbar_kws={"label": "Jaccard"})
axes[1].set_title("Pairwise overlap of selected interactions", fontweight="bold")
axes[1].set_xlabel("downsampling seed")
axes[1].set_ylabel("downsampling seed")

fig.suptitle("Pooled downsampling seed stability\\n"
             "sensitivity to which cells were drawn, NOT patient-level validity\\n"
             + POOLED_SUBTITLE,
             fontsize=FONT_SIZE, fontweight="bold")
fig.tight_layout()
seed_written = save_fig(fig, FIG_POOLED_SEEDS, "figB4_seed_stability",
                        list(seed_tables.values()), "exploratory_pooled/seed_stability")

seed_stability_mode = f"per_seed_wilcoxon_tables_n{len(seed_frames)}"
seed_summary = find_table(["pooled_downsampling_summary.csv", "*seed*.csv",
                           "*stability*.csv"], roots=[SENSITIVITY_DIR], required=False)

_log(event="seed_stability_figure", mode=seed_stability_mode,
     seeds=seeds_sorted, sources=[str(p) for p in seed_tables.values()],
     sensitivity_summary_from_06=str(seed_summary) if seed_summary else None,
     n_interactions_selected_in_any_seed=int(len(sel_only)),
     mean_pairwise_jaccard=float(jac.values[np.triu_indices(len(seeds_sorted), 1)].mean())
     if len(seeds_sorted) > 1 else None,
     outputs=[str(p) for p in seed_written] + [str(freq_path), str(jac_path)])

print(f"seed stability: {seed_stability_mode}")

In [ ]:
# =============================================================================
# SUCCESS RECORD
# =============================================================================
provenance_df = pd.DataFrame(FIGURE_PROVENANCE)
provenance_path = FIG_ROOT / "figure_provenance.csv"
provenance_df.to_csv(provenance_path, index=False)

all_figures = sorted(set(FIG_ROOT.rglob("*.png")) | set(FIG_ROOT.rglob("*.svg")))
all_tables  = sorted(TABLES_DIR.rglob("*.csv")) + sorted(TABLES_DIR.rglob("*.xlsx"))
all_outputs = sorted(set(all_figures + all_tables + [provenance_path]))

for p in all_outputs:
    assert p.is_file() and p.stat().st_size > 0, f"missing or empty output: {p}"
    assert str(p.resolve()).startswith(str(OUTPUT_ROOT.resolve())), \
        f"output escaped the new output root: {p}"

assert len(provenance_df) > 0
assert provenance_df["source_tables"].str.len().gt(0).all(), \
    "a figure was written without recording its source table"

_call_flex(write_manifest,
           ((all_outputs, MANIFEST_PATH), {}),
           ((MANIFEST_PATH, all_outputs), {}),
           ((FIG_ROOT, MANIFEST_PATH), {}),
           ((MANIFEST_PATH,), {"paths": all_outputs}))

SUCCESS = {
    "notebook": NOTEBOOK,
    "status": "success",
    "run_start_utc": RUN_START_ISO,
    "run_end_utc": datetime.now(timezone.utc).isoformat(),
    "elapsed_seconds": round(time.time() - RUN_START, 2),
    "config_sha256": CFG_SHA,
    "sections": {
        "primary_patient_level": {
            "n_tested_endpoints": int(len(tested)),
            "n_fdr_hits": n_hits,
            "forest_mode": forest_mode,
            "n_availability_flagged": n_flagged,
            "workbook_status": workbook_status,
            "workbook_path": str(workbook_path),
        },
        "exploratory_pooled": {
            "descriptive_runs": cpdb_plot_report,
            "differential_figures": [str(p) for p in diff_written],
            "consistency_interactions": int(n_consistency_total),
            "seed_stability_mode": seed_stability_mode,
            "subtitle_applied": POOLED_SUBTITLE,
        },
    },
    "deletion_safety": {
        "mechanism": "utils.safe_unlink(path, allowed_root)",
        "allowed_root": str(DELETION_ALLOWED_ROOT),
        "stale_files_removed": removed,
        "grep_subprocess_removed": True,
        "legacy_input_dir_eq_script_dir_removed": True,
    },
    "interaction_coverage": (
        json.loads((OUTPUT_ROOT / "audit" / "cpdb_interaction_coverage.json").read_text())
        ["interactions"]
        if (OUTPUT_ROOT / "audit" / "cpdb_interaction_coverage.json").is_file()
        else "coverage artifact not found"),
    "coverage_caveat": ("all interaction counts plotted are counts within the evaluable "
                        "subset; absence is not evidence of biological absence"),
    "n_figures": len(all_figures),
    "n_tables": len(all_tables),
    "figure_provenance": str(provenance_path),
    "output_sha256": {str(p): sha256_file(p) for p in all_outputs},
    "manifest": str(MANIFEST_PATH),
}
SUCCESS_JSON.write_text(json.dumps(SUCCESS, indent=2))
_log(event="notebook_success", **{k: v for k, v in SUCCESS.items() if k != "output_sha256"})
print(f"figures: {len(all_figures)}   tables: {len(all_tables)}")
SUCCESS["sections"]